# Agentic AI Investigation Team

A vendor-agnostic multi-agent framework for autonomous network investigation and root cause analysis.

---

## Notebook Sections

0. Project Setup
1. Imports
2. Shared Types
3. Configuration
4. Domain Model
5. Evidence Graph
6. Evidence Adapters
7. Deterministic Executors
8. AI Agents
9. Investigation Workflow
10. Validation

DESIGN PHILOSOPHY

Agents reason.

Executors execute.

Evidence adapters translate.

The Evidence Graph is the single source of truth.

Agents never interact directly with vendor-specific data.

#0. PROJECT SETUP

Connect to github, clone repo if necessary, etc.

In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import pprint

from google.colab import drive


# ============================================================
# PROJECT SETTINGS
# ============================================================

REPO_NAME = "agentic-ai-investigation-team"
GITHUB_USER = "icarovazquez"
BRANCH = "main"

DRIVE_MOUNT = Path("/content/drive")
PROJECTS_DIR = DRIVE_MOUNT / "MyDrive" / "Colab Notebooks"
PROJECT_ROOT = PROJECTS_DIR / REPO_NAME

# Store the persistent key in Google Drive.
DRIVE_SSH_DIR = DRIVE_MOUNT / "MyDrive" / ".ssh_colab"
DRIVE_PRIVATE_KEY = DRIVE_SSH_DIR / "id_ed25519"
DRIVE_PUBLIC_KEY = DRIVE_SSH_DIR / "id_ed25519.pub"

# Runtime SSH directory. Colab resets this when the runtime restarts.
RUNTIME_SSH_DIR = Path.home() / ".ssh"
RUNTIME_PRIVATE_KEY = RUNTIME_SSH_DIR / "id_ed25519"
RUNTIME_PUBLIC_KEY = RUNTIME_SSH_DIR / "id_ed25519.pub"
SSH_CONFIG = RUNTIME_SSH_DIR / "config"
KNOWN_HOSTS = RUNTIME_SSH_DIR / "known_hosts"

SSH_REPO_URL = (
    f"git@github.com:{GITHUB_USER}/{REPO_NAME}.git"
)


# ============================================================
# COMMAND HELPER
# ============================================================

def run_command(
    command: list[str],
    *,
    cwd: Path | None = None,
    check: bool = True,
) -> subprocess.CompletedProcess:
    """Run a shell command and display its output."""

    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
    )

    if result.stdout.strip():
        print(result.stdout.strip())

    if result.stderr.strip():
        print(result.stderr.strip())

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}:\n"
            f"{' '.join(command)}"
        )

    return result


# ============================================================
# 1. MOUNT GOOGLE DRIVE
# ============================================================

if not (DRIVE_MOUNT / "MyDrive").exists():
    drive.mount(str(DRIVE_MOUNT))
else:
    print("✓ Google Drive is already mounted.")


# ============================================================
# 2. CREATE A PERSISTENT SSH KEY IF NEEDED
# ============================================================

DRIVE_SSH_DIR.mkdir(parents=True, exist_ok=True)

new_key_created = False

if not DRIVE_PRIVATE_KEY.exists():
    print("Creating a persistent SSH key in Google Drive...")

    run_command(
        [
            "ssh-keygen",
            "-t",
            "ed25519",
            "-C",
            "icarovazquez-colab",
            "-f",
            str(DRIVE_PRIVATE_KEY),
            "-N",
            "",
        ]
    )

    new_key_created = True
    print("✓ Persistent SSH key created.")
else:
    print("✓ Persistent SSH key already exists in Google Drive.")


# ============================================================
# 3. COPY THE KEY INTO THE COLAB RUNTIME
# ============================================================

RUNTIME_SSH_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(DRIVE_PRIVATE_KEY, RUNTIME_PRIVATE_KEY)
shutil.copy2(DRIVE_PUBLIC_KEY, RUNTIME_PUBLIC_KEY)

os.chmod(RUNTIME_SSH_DIR, 0o700)
os.chmod(RUNTIME_PRIVATE_KEY, 0o600)
os.chmod(RUNTIME_PUBLIC_KEY, 0o644)

print("✓ SSH key copied into the Colab runtime.")


# ============================================================
# 4. CREATE SSH CONFIGURATION
# ============================================================

SSH_CONFIG.write_text(
    f"""Host github.com
    HostName github.com
    User git
    IdentityFile {RUNTIME_PRIVATE_KEY}
    IdentitiesOnly yes
"""
)

os.chmod(SSH_CONFIG, 0o600)

print("✓ SSH configuration created.")


# ============================================================
# 5. ADD GITHUB TO KNOWN_HOSTS
# ============================================================

keyscan = subprocess.run(
    ["ssh-keyscan", "-t", "ed25519", "github.com"],
    text=True,
    capture_output=True,
    check=True,
)

KNOWN_HOSTS.write_text(keyscan.stdout)
os.chmod(KNOWN_HOSTS, 0o644)

print("✓ GitHub added to known_hosts.")


# ============================================================
# FIRST-RUN GITHUB REGISTRATION
# ============================================================

if new_key_created:
    print("\n" + "=" * 70)
    print("ONE-TIME GITHUB SETUP REQUIRED")
    print("=" * 70)
    print(
        "Copy the public key below and add it at:\n"
        "GitHub → Settings → SSH and GPG keys → New SSH key\n"
    )

    print(DRIVE_PUBLIC_KEY.read_text().strip())

    print(
        "\nAfter adding the key to GitHub, rerun this cell."
    )

    raise SystemExit(
        "SSH key created. Add the public key to GitHub, then rerun."
    )


# ============================================================
# TEST GITHUB AUTHENTICATION
# ============================================================

ssh_test = run_command(
    [
        "ssh",
        "-o",
        "StrictHostKeyChecking=yes",
        "-T",
        "git@github.com",
    ],
    check=False,
)

# GitHub returns exit code 1 even when authentication succeeds.
ssh_output = (
    ssh_test.stdout + "\n" + ssh_test.stderr
).lower()

if "successfully authenticated" not in ssh_output:
    raise RuntimeError(
        "GitHub SSH authentication failed.\n"
        "Confirm that the displayed public key was added to your "
        "GitHub account under SSH and GPG keys."
    )

print("✓ GitHub SSH authentication succeeded.")


# ============================================================
# 6. CLONE OR UPDATE THE REPOSITORY
# ============================================================

PROJECTS_DIR.mkdir(parents=True, exist_ok=True)

if not PROJECT_ROOT.exists():
    print(f"Cloning repository into:\n{PROJECT_ROOT}")

    run_command(
        [
            "git",
            "clone",
            "--branch",
            BRANCH,
            "--single-branch",
            SSH_REPO_URL,
            str(PROJECT_ROOT),
        ]
    )

elif not (PROJECT_ROOT / ".git").exists():
    raise RuntimeError(
        f"{PROJECT_ROOT} exists but is not a Git repository."
    )

else:
    print(f"✓ Repository already exists at:\n{PROJECT_ROOT}")

    # Ensure the remote uses SSH rather than HTTPS.
    run_command(
        [
            "git",
            "remote",
            "set-url",
            "origin",
            SSH_REPO_URL,
        ],
        cwd=PROJECT_ROOT,
    )

    status = run_command(
        ["git", "status", "--porcelain"],
        cwd=PROJECT_ROOT,
    )

    if status.stdout.strip():
        print(
            "\nLocal changes detected. Git pull was skipped so that "
            "uncommitted work is not overwritten."
        )
    else:
        print(f"Pulling the latest origin/{BRANCH} changes...")

        run_command(
            [
                "git",
                "pull",
                "--ff-only",
                "origin",
                BRANCH,
            ],
            cwd=PROJECT_ROOT,
        )


# ============================================================
# ENTER THE PROJECT DIRECTORY
# ============================================================

os.chdir(PROJECT_ROOT)

print("\n" + "=" * 70)
print("PROJECT INITIALIZATION COMPLETE")
print("=" * 70)
print(f"Project root: {PROJECT_ROOT}")

run_command(["git", "remote", "-v"], cwd=PROJECT_ROOT)
run_command(["git", "status", "--short", "--branch"], cwd=PROJECT_ROOT)

Mounted at /content/drive
✓ Persistent SSH key already exists in Google Drive.
✓ SSH key copied into the Colab runtime.
✓ SSH configuration created.
✓ GitHub added to known_hosts.
Hi icarovazquez! You've successfully authenticated, but GitHub does not provide shell access.
✓ GitHub SSH authentication succeeded.
✓ Repository already exists at:
/content/drive/MyDrive/Colab Notebooks/agentic-ai-investigation-team
M "notebooks/Agentic AI Investigation Team.ipynb"

Local changes detected. Git pull was skipped so that uncommitted work is not overwritten.

PROJECT INITIALIZATION COMPLETE
Project root: /content/drive/MyDrive/Colab Notebooks/agentic-ai-investigation-team
origin	git@github.com:icarovazquez/agentic-ai-investigation-team.git (fetch)
origin	git@github.com:icarovazquez/agentic-ai-investigation-team.git (push)
## main...origin/main
 M "notebooks/Agentic AI Investigation Team.ipynb"


CompletedProcess(args=['git', 'status', '--short', '--branch'], returncode=0, stdout='## main...origin/main\n M "notebooks/Agentic AI Investigation Team.ipynb"\n', stderr='')

#1. IMPORTS & SHARED UTILITIES

In [2]:
!pip install --upgrade aisuite aisuite[anthropic] #Andrew Ng's wrapper lib that calls different LLM provides and abstracts each providers' API needs
!pip install anthropic #for calling anthropic LLMs through aisuite
!pip install -U "langfuse>2.0.0" #obervability & evals

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.2/228.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 16.0 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.63b1
    Uninstalling opentelemetry-semantic-conventions-0.63b1:
      Successfully uninstalled opentelemetry-semant

In [96]:
from __future__ import annotations

from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from enum import Enum
from pathlib import Path
from typing import Any, Dict, List, Optional
from uuid import uuid4
from abc import ABC, abstractmethod
from google.colab import userdata


import networkx as nx
import aisuite as ai
import ast
import re
import json

#initialize ai client
Client = ai.Client()

os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

DEFAULT_AGENT_MODEL = "anthropic:claude-haiku-4-5"

AGENT_MODEL_MAP = {
    "incident_framing_agent": DEFAULT_AGENT_MODEL,
    "hypothesis_generator_agent": DEFAULT_AGENT_MODEL,
    "evidence_planning_agent": DEFAULT_AGENT_MODEL,
    "evidence_analyst_agent": DEFAULT_AGENT_MODEL,
}

AGENT_MAX_TOKENS = {
    "incident_framing_agent": 2500,
    "hypothesis_generator_agent": 2000,
    "evidence_planning_agent": 6000,
    "evidence_analyst_agent": 2500,
}

LLM_USAGE = []

#2. OBSERVABILITY & LLMs

In [4]:
from langfuse import Langfuse, get_client
from langfuse import observe

LANGFUSE_PUBLIC_KEY = userdata.get("LANGFUSE_PUBLIC_KEY_I_TEAM").strip()
LANGFUSE_SECRET_KEY = userdata.get("LANGFUSE_SECRET_KEY_I_TEAM").strip()
LANGFUSE_BASE_URL = "https://cloud.langfuse.com"

langfuse = Langfuse(
    public_key=LANGFUSE_PUBLIC_KEY,
    secret_key=LANGFUSE_SECRET_KEY,
    base_url=LANGFUSE_BASE_URL,
)

try:
    projects = langfuse.api.projects.get()
    print("✅ Connected. Projects:", [p.name for p in projects.data])
except Exception as e:
    print("❌ Langfuse connection failed")
    print("Type:", type(e).__name__)
    print("Message:", str(e))

✅ Connected. Projects: ['agentic-a-investigation-team']


In [5]:
def trim_messages(messages, max_chars=12000):

    trimmed = []
    total_chars = 0

    # start from newest messages first
    for msg in reversed(messages):

        content = msg.get("content", "")

        if not isinstance(content, str):
            content = str(content)

        remaining = max_chars - total_chars

        if remaining <= 0:
            break

        # trim oversized message if needed
        if len(content) > remaining:
            content = content[-remaining:]

        trimmed.append({
            **msg,
            "content": content
        })

        total_chars += len(content)

    # restore chronological order
    return list(reversed(trimmed))

In [6]:
@observe(name="llm_call", as_type='generation')
def llm_call(agent_name:str, messages: list, temperature: float = 1.0, tools: list= None) -> str:

  """
  observability wrapper that all agents will use when calling the llm
  logs model, input, output, and token usage to Langfuse
  """

  selected_model = AGENT_MODEL_MAP.get(agent_name, DEFAULT_AGENT_MODEL)

  max_tokens = AGENT_MAX_TOKENS.get(agent_name, 3000)

  messages = trim_messages(messages, max_chars=12000)

  call_kwargs = {
    "model": selected_model,
    "messages": messages,
    "temperature": temperature,
    "max_tokens": max_tokens,
  }

  if tools:
    call_kwargs["tools"] = tools

  #calling the llm
  response = Client.chat.completions.create(**call_kwargs)
  content = response.choices[0].message.content

  #log output after calling the LLM
  usage = getattr(response, "usage", None)

  input_tokens = getattr(usage, "prompt_tokens", 0) if usage else 0
  output_tokens = getattr(usage, "completion_tokens", 0) if usage else 0
  total_tokens = input_tokens + output_tokens

  result = {
        "agent_name": agent_name,
        "model": selected_model,
        "temperature": temperature,
        "content": content,
        "usage": {
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": total_tokens
        } if usage else {},
        "tools_available": bool(tools),
    }

  LLM_USAGE.append({
    "agent": agent_name,
    "model": selected_model,
    "input_tokens": input_tokens,
    "output_tokens": output_tokens,
    "total_tokens": total_tokens})

  print('llm call completed')

  return result


In [7]:
def clean_llm_dict_output(raw_output: str) -> str:
    cleaned = raw_output.strip()

    if not cleaned:
        raise ValueError(
            "Agent returned an empty response."
        )

    if cleaned.startswith("```"):
        lines = cleaned.splitlines()

        if lines[0].strip().startswith("```"):
            lines = lines[1:]

        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]

        cleaned = "\n".join(lines).strip()

    start = cleaned.find("{")
    end = cleaned.rfind("}")

    if start == -1 or end == -1 or end < start:
        raise ValueError(
            "Agent did not return the required dictionary.\n\n"
            f"Raw output:\n{raw_output}"
        )

    return cleaned[start:end + 1]

In [8]:
def parse_agent_response(
    raw_output: str,
) -> Dict[str, Any]:

    cleaned = clean_llm_dict_output(
        raw_output
    )

    try:
        parsed = ast.literal_eval(
            cleaned
        )

    except (ValueError, SyntaxError) as exc:
        raise ValueError(
            "Unable to parse agent response as a Python dictionary.\n\n"
            f"Cleaned output:\n{cleaned}"
        ) from exc

    if not isinstance(parsed, dict):
        raise TypeError(
            "Agent response must evaluate to a Python dictionary."
        )

    return parsed

In [9]:
def repair_agent_dict_response(
    raw_output: str,
    expected_schema: str,
    agent_name: str,
) -> Dict[str, Any]:
    """
    Repair an LLM response that failed the structured-output contract.

    The repair call performs formatting only. It must preserve the
    original analysis and convert it into the required dictionary.
    """

    repair_prompt = f"""
You are a structured-output formatter.

The previous agent returned an answer that violated its required
output format.

Your ONLY job is to convert the supplied response into a Python
dictionary matching the required schema.

Do not:
- add new reasoning;
- add new evidence;
- determine root cause;
- recommend remediation;
- include Markdown;
- include headings;
- include prose outside the dictionary.

For hypothesis status, ONLY these values are valid:

- supported
- weakened
- rejected
- proposed

Map equivalent language as follows:

confirmed -> supported
strongly supported -> supported
unsupported -> weakened
falsified -> rejected
inconclusive -> proposed
unresolved -> proposed

Required schema:

{expected_schema}

Original response:

{raw_output}

Return ONLY the Python dictionary.
"""

    repair_response = llm_call(
        agent_name=f"{agent_name}_format_repair",
        messages=[
            {
                "role": "system",
                "content": (
                    "You convert malformed agent output into "
                    "strict structured Python dictionaries."
                ),
            },
            {
                "role": "user",
                "content": repair_prompt,
            },
        ],
        temperature=0.0,
    )

    return parse_agent_response(
        repair_response["content"]
    )

In [10]:
def parse_or_repair_agent_response(
    raw_output: str,
    expected_schema: str,
    agent_name: str,
) -> Dict[str, Any]:
    """
    Parse structured agent output.

    If the model violates the output contract, make one formatting-only
    repair attempt and parse the repaired result.

    Preserve hypothesis_id values EXACTLY as they appear in the original
    response or required input. Never abbreviate them to H1, H2, H3, etc.
    """

    try:
        return parse_agent_response(
            raw_output
        )

    except ValueError:
        print(
            f"⚠ {agent_name} returned malformed structured output. "
            "Attempting format repair."
        )

        return repair_agent_dict_response(
            raw_output=raw_output,
            expected_schema=expected_schema,
            agent_name=agent_name,
        )

In [11]:
EVIDENCE_ANALYSIS_SCHEMA = """
{
    "assessments": [
        {
            "hypothesis_id": str,
            "status": str,
            "confidence": float,
            "supporting_evidence": list[str],
            "contradicting_evidence": list[str],
            "missing_evidence": list[str],
            "rationale": str
        }
    ]
}
"""

HYPOTHESIS_SET_SCHEMA = """
{
    "hypotheses": [
        {
            "title": str,
            "proposed_cause": str,
            "causal_mechanism": str,
            "suspected_entity_ids": list[str],
            "expected_observations": list[str],
            "falsifying_observations": list[str],
            "required_capabilities": list[str],
            "prior_confidence": float
        }
    ]
}
"""

EVIDENCE_PLAN_SCHEMA = """
{
    "tests": [
        {
            "hypothesis_id": str,
            "objective": str,
            "capability": str,
            "parameters": dict,
            "expected_supporting_observations": list[str],
            "expected_falsifying_observations": list[str],
            "priority": int
        }
    ]
}
"""

# ============================================================
# ROOT CAUSE & REMEDIATION — SCHEMA
# ============================================================

ROOT_CAUSE_RECOMMENDATION_SCHEMA = """
{
    "root_cause_explanation": str,
    "remediation_steps": [
        {
            "action": str,
            "rationale": str,
            "risk_level": str
        }
    ],
    "residual_risk": str,
    "challenge_acknowledged": str
}
"""

#3. SHARED TYPES

In [12]:
from enum import Enum


class EntityType(str, Enum):
    """Types of entities that can appear in the Evidence Graph."""

    SITE = "site"
    DEVICE = "device"
    INTERFACE = "interface"
    LINK = "link"
    HOST = "host"
    SERVICE = "service"
    APPLICATION = "application"
    NETWORK = "network"
    VRF = "vrf"
    VLAN = "vlan"
    ROUTE = "route"


class RelationshipType(str, Enum):
    """Relationships between entities."""

    CONTAINS = "contains"
    CONNECTED_TO = "connected_to"
    DEPENDS_ON = "depends_on"
    ROUTES_TO = "routes_to"
    HOSTS = "hosts"
    MEMBER_OF = "member_of"
    PEERS_WITH = "peers_with"
    BACKS_UP = "backs_up"


class EvidenceType(str, Enum):
    """Evidence categories."""

    TOPOLOGY = "topology"
    TELEMETRY = "telemetry"
    CONFIGURATION = "configuration"
    EVENT = "event"
    CHANGE = "change"
    FLOW = "flow"
    SYNTHETIC_TEST = "synthetic_test"
    ROUTING = "routing"


class InvestigationStatus(str, Enum):
    """Overall investigation lifecycle."""

    CREATED = "created"
    IN_PROGRESS = "in_progress"
    WAITING_FOR_EVIDENCE = "waiting_for_evidence"
    ROOT_CAUSE_IDENTIFIED = "root_cause_identified"
    REMEDIATION_RECOMMENDED = "remediation_recommended"
    CLOSED = "closed"
    INCONCLUSIVE = "inconclusive"


class HypothesisStatus(str, Enum):
    """Status of an investigation hypothesis."""

    PROPOSED = "proposed"
    SUPPORTED = "supported"
    WEAKENED = "weakened"
    REJECTED = "rejected"
    CONFIRMED = "confirmed"


class EvidenceDirection(str, Enum):
    """How evidence affects a hypothesis."""

    SUPPORTS = "supports"
    CONTRADICTS = "contradicts"
    NEUTRAL = "neutral"


class RemediationMode(str, Enum):
    """How remediation is handled."""

    RECOMMEND_ONLY = "recommend_only"
    HUMAN_APPROVAL = "human_approval"
    AUTOMATIC = "automatic"


class SourceFormat(str, Enum):
    """Physical or logical format exposed by an evidence source."""

    CSV = "csv"
    JSON = "json"
    JSONL = "jsonl"
    PARQUET = "parquet"
    GRAPHML = "graphml"
    YAML = "yaml"
    API = "api"
    DATABASE = "database"
    COMMAND = "command"
    LIVE_LAB = "live_lab"


class AccessMode(str, Enum):
    """How an adapter accesses an evidence source."""

    FILE = "file"
    API = "api"
    DATABASE = "database"
    COMMAND = "command"
    LIVE_LAB = "live_lab"

#4. CONFIGURATION

The configuration layer describes one reproducible network investigation.

It defines:

- the incident seed;
- available evidence sources;
- source adapters and access methods;
- investigation policies;
- optional benchmark ground truth; and
- output settings.

Configuration objects describe the investigation environment. They do not load evidence or make diagnostic decisions.

##Incident Seed

In [13]:
@dataclass
class IncidentSeed:
    """
    Initial incident information supplied to the investigation.

    This is not the evolving IncidentRecord. It is the reproducible
    starting point used by the Incident Framing Agent.
    """

    reported_symptom: str
    incident_start_time: str

    affected_service: Optional[str] = None
    reported_locations: List[str] = field(default_factory=list)
    reported_entities: List[str] = field(default_factory=list)

    source: str = "benchmark"
    initial_severity: Optional[str] = None
    initial_context: Dict[str, Any] = field(default_factory=dict)

    def validate(self) -> None:
        if not self.reported_symptom.strip():
            raise ValueError("reported_symptom cannot be empty.")

        if not self.incident_start_time.strip():
            raise ValueError("incident_start_time cannot be empty.")

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

##Evidence Source Configuration

In [14]:
@dataclass
class EvidenceSourceConfig:
    """
    Configuration for one raw evidence source.

    The adapter_name identifies the translator that converts the raw
    source into vendor-neutral Evidence Graph domain records.
    """

    source_id: str
    evidence_type: EvidenceType
    source_format: SourceFormat
    access_mode: AccessMode
    adapter_name: str

    path: Optional[str] = None
    url: Optional[str] = None

    connection_parameters: Dict[str, Any] = field(
        default_factory=dict
    )
    query_parameters: Dict[str, Any] = field(
        default_factory=dict
    )

    timestamp_column: Optional[str] = None
    entity_id_columns: List[str] = field(
        default_factory=list
    )
    schema_mapping: Dict[str, str] = field(
        default_factory=dict
    )

    required: bool = True
    enabled: bool = True

    metadata: Dict[str, Any] = field(
        default_factory=dict
    )

    def validate(self) -> None:
        if not self.source_id.strip():
            raise ValueError("source_id cannot be empty.")

        if not self.adapter_name.strip():
            raise ValueError(
                f"Source '{self.source_id}' requires adapter_name."
            )

        if (
            self.access_mode == AccessMode.FILE
            and not self.path
        ):
            raise ValueError(
                f"File source '{self.source_id}' requires a path."
            )

        if (
            self.access_mode == AccessMode.API
            and not self.url
            and not self.connection_parameters
        ):
            raise ValueError(
                f"API source '{self.source_id}' requires a URL "
                "or connection parameters."
            )

        if self.access_mode in {
            AccessMode.DATABASE,
            AccessMode.COMMAND,
            AccessMode.LIVE_LAB,
        } and not self.connection_parameters:
            raise ValueError(
                f"Source '{self.source_id}' requires "
                "connection_parameters."
            )

    def to_dict(self) -> Dict[str, Any]:
        result = asdict(self)

        result["evidence_type"] = self.evidence_type.value
        result["source_format"] = self.source_format.value
        result["access_mode"] = self.access_mode.value

        return result

##Investigation Policies

In [15]:
@dataclass
class InvestigationPolicy:
    """
    Guardrails and stopping conditions for the investigation workflow.
    """

    remediation_mode: RemediationMode = (
        RemediationMode.RECOMMEND_ONLY
    )

    require_human_approval: bool = True
    minimum_diagnosis_confidence: float = 0.80

    maximum_reasoning_iterations: int = 5
    maximum_hypotheses: int = 5
    maximum_tests_per_hypothesis: int = 5

    require_falsifiable_hypotheses: bool = True
    require_challenger_review: bool = True
    require_evidence_references: bool = True

    def validate(self) -> None:
        if not 0.0 <= self.minimum_diagnosis_confidence <= 1.0:
            raise ValueError(
                "minimum_diagnosis_confidence must be "
                "between 0 and 1."
            )

        if self.maximum_reasoning_iterations < 1:
            raise ValueError(
                "maximum_reasoning_iterations must be at least 1."
            )

        if self.maximum_hypotheses < 2:
            raise ValueError(
                "maximum_hypotheses must be at least 2."
            )

        if self.maximum_tests_per_hypothesis < 1:
            raise ValueError(
                "maximum_tests_per_hypothesis must be at least 1."
            )

    def to_dict(self) -> Dict[str, Any]:
        result = asdict(self)
        result["remediation_mode"] = self.remediation_mode.value
        return result

##Evaluation Configuration

In [16]:
@dataclass
class EvaluationConfig:
    """
    Optional benchmark ground truth and evaluation metrics.

    Production incidents may not have ground truth, so this object
    is optional.
    """

    ground_truth_root_cause: Optional[str] = None
    ground_truth_entity_ids: List[str] = field(
        default_factory=list
    )
    ground_truth_fault_type: Optional[str] = None

    metrics: List[str] = field(
        default_factory=lambda: [
            "top_1_root_cause_accuracy",
            "top_3_root_cause_accuracy",
            "localization_accuracy",
            "evidence_grounding_score",
            "investigation_efficiency",
        ]
    )

    @property
    def has_ground_truth(self) -> bool:
        return self.ground_truth_root_cause is not None

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

In [17]:
@dataclass
class NetworkInvestigationConfig:
    """
    Top-level configuration for one reproducible network
    investigation.

    This is the network-investigation equivalent of CompetitionConfig
    in the Agentic AI Data Science Team.
    """

    investigation_id: str
    investigation_name: str
    environment_name: str
    investigation_type: str

    scenario_provider: str
    scenario_id: str

    incident_seed: IncidentSeed
    evidence_sources: List[EvidenceSourceConfig]

    twin_source_id: Optional[str] = None

    investigation_policy: InvestigationPolicy = field(
        default_factory=InvestigationPolicy
    )

    evaluation_config: Optional[EvaluationConfig] = None

    output_dir: str = "/content/network_investigations"

    metadata: Dict[str, Any] = field(
        default_factory=dict
    )

    @property
    def enabled_sources(self) -> List[EvidenceSourceConfig]:
        return [
            source
            for source in self.evidence_sources
            if source.enabled
        ]

    @property
    def required_sources(self) -> List[EvidenceSourceConfig]:
        return [
            source
            for source in self.enabled_sources
            if source.required
        ]

    @property
    def available_evidence_types(self) -> List[str]:
        return sorted(
            {
                source.evidence_type.value
                for source in self.enabled_sources
            }
        )

    @property
    def investigation_output_dir(self) -> Path:
        return Path(self.output_dir) / self.investigation_id

    @property
    def report_path(self) -> Path:
        return (
            self.investigation_output_dir
            / "investigation_report.json"
        )

    @property
    def history_path(self) -> Path:
        return (
            self.investigation_output_dir
            / "investigation_history.json"
        )

    def get_source(
        self,
        source_id: str,
    ) -> EvidenceSourceConfig:
        for source in self.enabled_sources:
            if source.source_id == source_id:
                return source

        raise KeyError(
            f"Evidence source '{source_id}' does not exist "
            "or is disabled."
        )

    def get_sources_by_type(
        self,
        evidence_type: EvidenceType,
    ) -> List[EvidenceSourceConfig]:
        return [
            source
            for source in self.enabled_sources
            if source.evidence_type == evidence_type
        ]

    def validate(self) -> None:
        if not self.investigation_id.strip():
            raise ValueError(
                "investigation_id cannot be empty."
            )

        if not self.investigation_name.strip():
            raise ValueError(
                "investigation_name cannot be empty."
            )

        if not self.scenario_provider.strip():
            raise ValueError(
                "scenario_provider cannot be empty."
            )

        if not self.scenario_id.strip():
            raise ValueError(
                "scenario_id cannot be empty."
            )

        self.incident_seed.validate()
        self.investigation_policy.validate()

        source_ids = [
            source.source_id
            for source in self.enabled_sources
        ]

        duplicate_source_ids = sorted(
            {
                source_id
                for source_id in source_ids
                if source_ids.count(source_id) > 1
            }
        )

        if duplicate_source_ids:
            raise ValueError(
                "Duplicate evidence source IDs: "
                f"{duplicate_source_ids}"
            )

        for source in self.enabled_sources:
            source.validate()

        if self.twin_source_id is not None:
            twin_source = self.get_source(
                self.twin_source_id
            )

            if twin_source.evidence_type not in {
                EvidenceType.TOPOLOGY,
            }:
                raise ValueError(
                    f"twin_source_id '{self.twin_source_id}' "
                    "must reference a topology source."
                )

    def to_dict(self) -> Dict[str, Any]:
        return {
            "investigation_id": self.investigation_id,
            "investigation_name": self.investigation_name,
            "environment_name": self.environment_name,
            "investigation_type": self.investigation_type,
            "scenario_provider": self.scenario_provider,
            "scenario_id": self.scenario_id,
            "incident_seed": self.incident_seed.to_dict(),
            "evidence_sources": [
                source.to_dict()
                for source in self.evidence_sources
            ],
            "twin_source_id": self.twin_source_id,
            "available_evidence_types": (
                self.available_evidence_types
            ),
            "investigation_policy": (
                self.investigation_policy.to_dict()
            ),
            "evaluation_config": (
                self.evaluation_config.to_dict()
                if self.evaluation_config
                else None
            ),
            "output_dir": self.output_dir,
            "report_path": str(self.report_path),
            "history_path": str(self.history_path),
            "metadata": self.metadata,
        }

    def __repr__(self) -> str:
        return (
            "NetworkInvestigationConfig("
            f"id='{self.investigation_id}', "
            f"provider='{self.scenario_provider}', "
            f"scenario='{self.scenario_id}', "
            f"sources={len(self.enabled_sources)})"
        )

##EVALUATION RESULT

In [18]:
# ============================================================
# GROUND TRUTH EVALUATION
# ============================================================

@dataclass
class EvaluationResult:
    """
    Comparison of an investigation's outcome against benchmark
    ground truth, where available.

    Not all metrics in EvaluationConfig.metrics are computed here.
    localization_accuracy is a deterministic set comparison over
    entity IDs. top_1/top_3_root_cause_accuracy are keyword-match
    heuristics against free-text ground_truth_root_cause / fault
    type, since that field has no structured form to compare
    against exactly — treat these two as approximate signals, not
    ground truth verification. evidence_grounding_score and
    investigation_efficiency are not computed by this function (see
    note in compute_evaluation docstring).
    """

    investigation_id: str
    has_ground_truth: bool

    top_1_root_cause_accuracy: Optional[bool] = None
    top_3_root_cause_accuracy: Optional[bool] = None
    localization_accuracy: Optional[float] = None

    matched_entity_ids: List[str] = field(default_factory=list)
    missed_entity_ids: List[str] = field(default_factory=list)
    unexpected_entity_ids: List[str] = field(default_factory=list)

    notes: List[str] = field(default_factory=list)

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


def compute_evaluation(
    run_result: InvestigationRunResult,
    evaluation_config: Optional[EvaluationConfig],
) -> EvaluationResult:
    """
    Score a completed investigation run against benchmark ground
    truth, if present.

    Only computes localization_accuracy (deterministic) and
    top_1/top_3_root_cause_accuracy (keyword-match heuristic against
    ground_truth_fault_type — approximate, since ground_truth_root_cause
    has no structured form). evidence_grounding_score and
    investigation_efficiency from EvaluationConfig.metrics are not
    computed here: grounding would require validating every citation
    in supporting_evidence against evidence_results, and efficiency
    needs LLM_USAGE token/call accounting — both worth building
    separately once there's a reason to trust the numbers, rather
    than stubbing something that looks precise but isn't.
    """

    investigation_id = run_result.investigation_id

    if evaluation_config is None or not evaluation_config.has_ground_truth:
        return EvaluationResult(
            investigation_id=investigation_id,
            has_ground_truth=False,
            notes=["No ground truth configured for this investigation."],
        )

    recommendation = run_result.root_cause_recommendation

    if recommendation is None or recommendation.status != "root_cause_identified":
        return EvaluationResult(
            investigation_id=investigation_id,
            has_ground_truth=True,
            top_1_root_cause_accuracy=False,
            top_3_root_cause_accuracy=False,
            localization_accuracy=0.0,
            notes=[
                "No root cause was identified by the investigation "
                "(insufficient_evidence or earlier stage failure) — "
                "cannot match against ground truth."
            ],
        )

    hypotheses_by_id = {
        h.hypothesis_id: h
        for h in (run_result.hypothesis_set.hypotheses if run_result.hypothesis_set else [])
    }

    winning_hypothesis = hypotheses_by_id.get(recommendation.hypothesis_id)

    # --- localization_accuracy: deterministic entity-ID overlap ---

    ground_truth_ids = set(evaluation_config.ground_truth_entity_ids)
    predicted_ids = set(
        winning_hypothesis.suspected_entity_ids if winning_hypothesis else []
    )

    matched = sorted(ground_truth_ids & predicted_ids)
    missed = sorted(ground_truth_ids - predicted_ids)
    unexpected = sorted(predicted_ids - ground_truth_ids)

    if ground_truth_ids:
        localization_accuracy = len(matched) / len(ground_truth_ids)
    else:
        localization_accuracy = None

    # --- top_1 / top_3 root_cause_accuracy: keyword-match heuristic ---

    fault_type = (evaluation_config.ground_truth_fault_type or "").lower().replace("_", " ")
    root_cause_text = (evaluation_config.ground_truth_root_cause or "").lower()

    def hypothesis_matches_ground_truth(assessment_hypothesis_id: str) -> bool:
        hypothesis = hypotheses_by_id.get(assessment_hypothesis_id)
        if hypothesis is None:
            return False

        haystack = " ".join([
            hypothesis.title,
            hypothesis.proposed_cause,
            hypothesis.causal_mechanism,
        ]).lower()

        return (
            (fault_type and fault_type in haystack)
            or (root_cause_text and root_cause_text in haystack)
        )

    top_1_match = hypothesis_matches_ground_truth(recommendation.hypothesis_id)

    ranked = (
        rank_hypothesis_assessments(run_result.evidence_analysis)
        if run_result.evidence_analysis else []
    )
    top_3_ids = [a.hypothesis_id for a in ranked[:3]]
    top_3_match = any(hypothesis_matches_ground_truth(hid) for hid in top_3_ids)

    notes = [
        "top_1/top_3_root_cause_accuracy are keyword-match heuristics "
        "against free-text ground truth, not exact verification.",
    ]

    if not ground_truth_ids:
        notes.append(
            "ground_truth_entity_ids is empty — localization_accuracy "
            "not computed."
        )

    return EvaluationResult(
        investigation_id=investigation_id,
        has_ground_truth=True,
        top_1_root_cause_accuracy=top_1_match,
        top_3_root_cause_accuracy=top_3_match,
        localization_accuracy=localization_accuracy,
        matched_entity_ids=matched,
        missed_entity_ids=missed,
        unexpected_entity_ids=unexpected,
        notes=notes,
    )

##Synthetic Test

In [19]:
test_investigation_config = NetworkInvestigationConfig(
    investigation_id="synthetic-link-failure-001",
    investigation_name="Synthetic Link Failure",
    environment_name="development_fixture",
    investigation_type="connectivity_failure",

    scenario_provider="synthetic",
    scenario_id="link_failure_001",

    incident_seed=IncidentSeed(
        reported_symptom=(
            "The client host cannot reach the destination service."
        ),
        incident_start_time="2026-08-04T16:00:00Z",
        affected_service="service-app-01",
        reported_locations=["site-sfo"],
        reported_entities=["host-client-01"],
        source="synthetic_fixture",
        initial_severity="high",
    ),

    twin_source_id="synthetic_topology",

    evidence_sources=[
        EvidenceSourceConfig(
            source_id="synthetic_topology",
            evidence_type=EvidenceType.TOPOLOGY,
            source_format=SourceFormat.JSON,
            access_mode=AccessMode.FILE,
            adapter_name="SyntheticEvidenceAdapter",
            path="/content/data/synthetic/topology.json",
            required=True,
        ),
        EvidenceSourceConfig(
            source_id="synthetic_telemetry",
            evidence_type=EvidenceType.TELEMETRY,
            source_format=SourceFormat.CSV,
            access_mode=AccessMode.FILE,
            adapter_name="SyntheticTelemetryAdapter",
            path="/content/data/synthetic/telemetry.csv",
            required=False,
        ),
    ],

    investigation_policy=InvestigationPolicy(
        remediation_mode=RemediationMode.RECOMMEND_ONLY,
        require_human_approval=True,
        minimum_diagnosis_confidence=0.80,
        maximum_reasoning_iterations=5,
        maximum_hypotheses=5,
        maximum_tests_per_hypothesis=5,
    ),

    evaluation_config=EvaluationConfig(
        ground_truth_root_cause="Link failure between R1 and R2",
        ground_truth_entity_ids=["link-r1-r2"],
        ground_truth_fault_type="link_failure",
    ),

    metadata={
        "purpose": "configuration_contract_validation",
    },
)

test_investigation_config.validate()

print(test_investigation_config)
print()
print(
    "Enabled sources:",
    [
        source.source_id
        for source in test_investigation_config.enabled_sources
    ],
)
print(
    "Evidence types:",
    test_investigation_config.available_evidence_types,
)
print(
    "Report path:",
    test_investigation_config.report_path,
)

NetworkInvestigationConfig(id='synthetic-link-failure-001', provider='synthetic', scenario='link_failure_001', sources=2)

Enabled sources: ['synthetic_topology', 'synthetic_telemetry']
Evidence types: ['telemetry', 'topology']
Report path: /content/network_investigations/synthetic-link-failure-001/investigation_report.json


#5. DOMAIN MODEL

In [20]:
@dataclass
class NetworkEntity:
    entity_id: str
    entity_type: EntityType
    name: str

    attributes: Dict[str, Any] = field(default_factory=dict)
    source_ids: List[str] = field(default_factory=list)

    def validate(self) -> None:
        if not self.entity_id.strip():
            raise ValueError("entity_id cannot be empty.")

        if not self.name.strip():
            raise ValueError("name cannot be empty.")

    def to_dict(self) -> Dict[str, Any]:
        result = asdict(self)
        result["entity_type"] = self.entity_type.value
        return result


@dataclass
class NetworkRelationship:
    relationship_id: str
    source_entity_id: str
    target_entity_id: str
    relationship_type: RelationshipType

    attributes: Dict[str, Any] = field(default_factory=dict)

    valid_from: Optional[str] = None
    valid_to: Optional[str] = None

    source_ids: List[str] = field(default_factory=list)

    def validate(self) -> None:
        if not self.relationship_id.strip():
            raise ValueError("relationship_id cannot be empty.")

        if not self.source_entity_id.strip():
            raise ValueError("source_entity_id cannot be empty.")

        if not self.target_entity_id.strip():
            raise ValueError("target_entity_id cannot be empty.")

        if self.source_entity_id == self.target_entity_id:
            raise ValueError(
                "A relationship cannot connect an entity to itself."
            )

    def to_dict(self) -> Dict[str, Any]:
        result = asdict(self)
        result["relationship_type"] = self.relationship_type.value
        return result

In [21]:
@dataclass
class ObservationRecord:
    observation_id: str
    source_id: str
    entity_id: str

    metric_name: str
    metric_value: Any
    observed_at: str

    unit: Optional[str] = None
    dimensions: Dict[str, Any] = field(default_factory=dict)

    quality_score: float = 1.0

    def validate(self) -> None:
        if not self.observation_id.strip():
            raise ValueError("observation_id cannot be empty.")

        if not self.entity_id.strip():
            raise ValueError("entity_id cannot be empty.")

        if not self.metric_name.strip():
            raise ValueError("metric_name cannot be empty.")

        if not 0.0 <= self.quality_score <= 1.0:
            raise ValueError(
                "quality_score must be between 0 and 1."
            )

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass
class EventRecord:
    event_id: str
    source_id: str
    event_type: str
    occurred_at: str

    entity_ids: List[str] = field(default_factory=list)

    severity: Optional[str] = None
    message: Optional[str] = None
    attributes: Dict[str, Any] = field(default_factory=dict)

    def validate(self) -> None:
        if not self.event_id.strip():
            raise ValueError("event_id cannot be empty.")

        if not self.event_type.strip():
            raise ValueError("event_type cannot be empty.")

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass
class ChangeRecord:
    change_id: str
    source_id: str
    changed_at: str
    change_type: str

    entity_ids: List[str] = field(default_factory=list)

    previous_state: Any = None
    new_state: Any = None

    actor: Optional[str] = None
    approved: Optional[bool] = None
    rollback_reference: Optional[str] = None

    attributes: Dict[str, Any] = field(default_factory=dict)

    def validate(self) -> None:
        if not self.change_id.strip():
            raise ValueError("change_id cannot be empty.")

        if not self.change_type.strip():
            raise ValueError("change_type cannot be empty.")

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

##Incident Framing Agent Output

In [22]:
@dataclass
class IncidentFrame:
    """
    Structured output produced by the Incident Framing Agent.

    It converts the raw incident seed into a clear investigation
    problem without attempting to diagnose root cause.
    """

    incident_id: str

    investigation_question: str
    scope_summary: str

    known_facts: List[str] = field(default_factory=list)
    assumptions: List[str] = field(default_factory=list)
    unknowns: List[str] = field(default_factory=list)

    affected_entity_ids: List[str] = field(default_factory=list)
    potentially_relevant_entity_ids: List[str] = field(default_factory=list)

    required_capabilities: List[str] = field(default_factory=list)

    severity: Optional[str] = None

    def validate(self) -> None:
        if not self.incident_id.strip():
            raise ValueError("incident_id cannot be empty.")

        if not self.investigation_question.strip():
            raise ValueError(
                "investigation_question cannot be empty."
            )

        if not self.scope_summary.strip():
            raise ValueError(
                "scope_summary cannot be empty."
            )

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

In [23]:
def build_incident_framing_context(
    investigation_config: NetworkInvestigationConfig,
    evidence_graph: EvidenceGraph,
    topology_capability: Optional[TopologyCapability] = None,
) -> Dict[str, Any]:
    """
    Build the bounded context supplied to the Incident Framing Agent.

    The agent receives normalized evidence and capability output,
    never raw source data.
    """

    seed = investigation_config.incident_seed

    reported_entities = []

    for entity_id in seed.reported_entities:
        try:
            reported_entities.append(
                evidence_graph.get_entity(entity_id).to_dict()
            )

        except KeyError:
            reported_entities.append(
                {
                    "entity_id": entity_id,
                    "status": "not_found_in_evidence_graph",
                }
            )

    topology_evidence = None

    if (
        topology_capability is not None
        and len(seed.reported_entities) >= 2
    ):
        topology_evidence = topology_capability.collect_evidence(
            evidence_graph=evidence_graph,
            source_entity_id=seed.reported_entities[0],
            target_entity_id=seed.reported_entities[1],
        )

    # Vendor-reported incidents already known about this investigation.
    # Surfaced explicitly so the framing agent treats "what has a
    # monitoring vendor already told us" as a known fact from the
    # start, rather than something only reachable if a later
    # hypothesis happens to request vendor_alert evidence.
    vendor_alerts = [
        event.to_dict()
        for event in evidence_graph.get_events(
            event_types=["vendor_incident"]
        )
    ]

    return {
        "investigation_id": (
            investigation_config.investigation_id
        ),
        "investigation_name": (
            investigation_config.investigation_name
        ),
        "investigation_type": (
            investigation_config.investigation_type
        ),

        "reported_symptom": seed.reported_symptom,
        "incident_start_time": seed.incident_start_time,
        "affected_service": seed.affected_service,
        "reported_locations": list(
            seed.reported_locations
        ),
        "reported_entities": reported_entities,
        "initial_severity": seed.initial_severity,

        "available_evidence_types": (
            investigation_config.available_evidence_types
        ),

        "evidence_graph_summary": (
            evidence_graph.summary()
        ),

        "topology_evidence": topology_evidence,

        "vendor_alerts": vendor_alerts,

        "available_capabilities": [
            "topology",
            "reachability",
            "network_state",
            "vendor_alert",
        ],
    }

##Hypothesis Domain Object

In [24]:
@dataclass
class HypothesisRecord:
    """
    One falsifiable explanation for the reported incident.
    """

    hypothesis_id: str
    incident_id: str

    title: str
    proposed_cause: str
    causal_mechanism: str

    suspected_entity_ids: List[str] = field(
        default_factory=list
    )

    expected_observations: List[str] = field(
        default_factory=list
    )

    falsifying_observations: List[str] = field(
        default_factory=list
    )

    required_capabilities: List[str] = field(
        default_factory=list
    )

    prior_confidence: float = 0.0

    status: HypothesisStatus = (
        HypothesisStatus.PROPOSED
    )

    def validate(self) -> None:
        if not self.hypothesis_id.strip():
            raise ValueError(
                "hypothesis_id cannot be empty."
            )

        if not self.title.strip():
            raise ValueError(
                "Hypothesis title cannot be empty."
            )

        if not self.proposed_cause.strip():
            raise ValueError(
                "proposed_cause cannot be empty."
            )

        if not self.causal_mechanism.strip():
            raise ValueError(
                "causal_mechanism cannot be empty."
            )

        if not self.falsifying_observations:
            raise ValueError(
                "Every hypothesis must define at least "
                "one falsifying observation."
            )

        if not 0.0 <= self.prior_confidence <= 1.0:
            raise ValueError(
                "prior_confidence must be between 0 and 1."
            )

    def to_dict(self) -> Dict[str, Any]:
        result = asdict(self)
        result["status"] = self.status.value
        return result

In [25]:
@dataclass
class HypothesisSet:
    """
    Competing hypotheses generated for one incident.
    """

    incident_id: str
    hypotheses: List[HypothesisRecord]

    def validate(self) -> None:
        if len(self.hypotheses) < 2:
            raise ValueError(
                "At least two competing hypotheses "
                "are required."
            )

        hypothesis_ids = [
            hypothesis.hypothesis_id
            for hypothesis in self.hypotheses
        ]

        if len(hypothesis_ids) != len(
            set(hypothesis_ids)
        ):
            raise ValueError(
                "Hypothesis IDs must be unique."
            )

        for hypothesis in self.hypotheses:
            hypothesis.validate()

    def to_dict(self) -> Dict[str, Any]:
        return {
            "incident_id": self.incident_id,
            "hypotheses": [
                hypothesis.to_dict()
                for hypothesis in self.hypotheses
            ],
        }

##Hypothesis Generator Context

In [26]:
def build_hypothesis_generation_context(
    incident_frame: IncidentFrame,
    evidence_graph: EvidenceGraph,
    topology_capability: TopologyCapability,
) -> Dict[str, Any]:
    """
    Build bounded evidence context for hypothesis generation.
    """

    topology_evidence = None

    if len(
        incident_frame.affected_entity_ids
    ) >= 2:
        topology_evidence = (
            topology_capability.collect_evidence(
                evidence_graph=evidence_graph,
                source_entity_id=(
                    incident_frame.affected_entity_ids[0]
                ),
                target_entity_id=(
                    incident_frame.affected_entity_ids[1]
                ),
            )
        )

    return {
        "incident_frame": (
            incident_frame.to_dict()
        ),

        "topology_evidence": (
            topology_evidence
        ),

        "available_capabilities": [
            "topology",
            "reachability",
            "network_state",
        ],
    }

##Evidence Test Contracts

In [27]:
@dataclass
class EvidenceTest:
    """
    One planned evidence request for evaluating a hypothesis.
    """

    test_id: str
    hypothesis_id: str

    objective: str
    capability: str

    parameters: Dict[str, Any] = field(default_factory=dict)

    expected_supporting_observations: List[str] = field(
        default_factory=list
    )

    expected_falsifying_observations: List[str] = field(
        default_factory=list
    )

    priority: int = 1

    def validate(self) -> None:
        if not self.test_id.strip():
            raise ValueError("test_id cannot be empty.")

        if not self.hypothesis_id.strip():
            raise ValueError("hypothesis_id cannot be empty.")

        if not self.objective.strip():
            raise ValueError("objective cannot be empty.")

        if not self.capability.strip():
            raise ValueError("capability cannot be empty.")

        if self.priority < 1:
            raise ValueError("priority must be at least 1.")

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass
class EvidencePlan:
    """
    Planned deterministic evidence tests for one incident.
    """

    incident_id: str
    tests: List[EvidenceTest]

    def validate(self) -> None:
        if not self.incident_id.strip():
            raise ValueError("incident_id cannot be empty.")

        if not self.tests:
            raise ValueError(
                "EvidencePlan must contain at least one test."
            )

        test_ids = [test.test_id for test in self.tests]

        if len(test_ids) != len(set(test_ids)):
            raise ValueError("Evidence test IDs must be unique.")

        for test in self.tests:
            test.validate()

    def to_dict(self) -> Dict[str, Any]:
        return {
            "incident_id": self.incident_id,
            "tests": [
                test.to_dict()
                for test in self.tests
            ],
        }

##Evidence Planning Context

In [28]:
def build_evidence_planning_context(
    incident_frame: IncidentFrame,
    hypothesis_set: HypothesisSet,
    prior_evidence_analysis: Optional[EvidenceAnalysis] = None,
    prior_challenge_report: Optional[ChallengeReport] = None,
) -> Dict[str, Any]:
    """
    Build the bounded context supplied to the Evidence Planning Agent.

    If prior_evidence_analysis / prior_challenge_report are supplied,
    this is a subsequent reasoning-loop round: the context includes
    what's still missing so far, so planning targets those gaps
    specifically rather than re-deriving the same tests as round 1.
    """

    context: Dict[str, Any] = {
        "incident_frame": incident_frame.to_dict(),

        "hypotheses": [
            hypothesis.to_dict()
            for hypothesis in hypothesis_set.hypotheses
        ],

        "available_capabilities": [
            "topology",
            "reachability",
            "network_state",
            "vendor_alert",
            "deep_diagnostics",
        ],

        "capability_descriptions": {
            "topology": (
                "Provides entity relationships, neighbors, paths, "
                "and structural blast-radius evidence."
            ),
            "reachability": (
                "Provides end-to-end or hop-level reachability "
                "evidence such as ping, loss, latency, or path tests."
            ),
            "network_state": (
                "Provides operational state such as interface status, "
                "device availability, routing adjacency state, and "
                "route presence when available."
            ),
            "vendor_alert": (
                "Provides externally reported incidents/alerts from "
                "monitoring or alerting vendors (e.g. PagerDuty). "
                "Describes what a vendor detected and reported, not "
                "the underlying root cause — treat as symptom-level "
                "evidence, same as any other observation."
            ),
            "deep_diagnostics": (
                "Provides device-level diagnostics NOT included in "
                "routine first-response telemetry: BGP session "
                "state, interface error counters, and administrative "
                "(configured) interface status. Use this capability "
                "specifically when network_state evidence alone "
                "cannot distinguish between competing causes (e.g. "
                "physical failure vs. protocol-driven state changes)."
            ),
        },
    }

    if prior_evidence_analysis is not None:
        context["prior_round_missing_evidence"] = [
            {
                "hypothesis_id": assessment.hypothesis_id,
                "status": assessment.status.value,
                "confidence": assessment.confidence,
                "missing_evidence": assessment.missing_evidence,
            }
            for assessment in prior_evidence_analysis.assessments
            if assessment.missing_evidence
        ]

    if prior_challenge_report is not None:
        context["prior_round_challenger_gaps"] = [
            {
                "hypothesis_id": challenge.hypothesis_id,
                "challenge_outcome": challenge.challenge_outcome,
                "additional_evidence_needed": challenge.additional_evidence_needed,
            }
            for challenge in prior_challenge_report.challenges
            if challenge.additional_evidence_needed
        ]

    return context

##Hypothesis Assessment

In [29]:
@dataclass
class HypothesisAssessment:
    hypothesis_id: str

    status: HypothesisStatus
    confidence: float

    supporting_evidence: List[str] = field(default_factory=list)
    contradicting_evidence: List[str] = field(default_factory=list)
    missing_evidence: List[str] = field(default_factory=list)

    rationale: str = ""

    def validate(self) -> None:
        if not self.hypothesis_id.strip():
            raise ValueError("hypothesis_id cannot be empty.")

        if not 0.0 <= self.confidence <= 1.0:
            raise ValueError("confidence must be between 0 and 1.")

        if not self.rationale.strip():
            raise ValueError("rationale cannot be empty.")

    def to_dict(self) -> Dict[str, Any]:
        result = asdict(self)
        result["status"] = self.status.value
        return result

In [30]:
@dataclass
class EvidenceAnalysis:
    incident_id: str
    assessments: List[HypothesisAssessment]

    def validate(self) -> None:
        if not self.assessments:
            raise ValueError(
                "EvidenceAnalysis must contain assessments."
            )

        for assessment in self.assessments:
            assessment.validate()

    def to_dict(self) -> Dict[str, Any]:
        return {
            "incident_id": self.incident_id,
            "assessments": [
                assessment.to_dict()
                for assessment in self.assessments
            ],
        }

##Evidence Analysis Context

In [31]:
def build_evidence_analysis_context(
    hypothesis_set: HypothesisSet,
    evidence_plan: EvidencePlan,
    evidence_results: List[EvidenceTestResult],
) -> Dict[str, Any]:
    """
    Build a compact hypothesis-specific evidence package.

    The Evidence Analyst receives only evidence relevant to each
    hypothesis rather than complete executor/capability payloads.
    """

    hypotheses_by_id = {
        hypothesis.hypothesis_id: hypothesis
        for hypothesis in hypothesis_set.hypotheses
    }

    tests_by_id = {
        test.test_id: test
        for test in evidence_plan.tests
    }

    evidence_by_hypothesis = {}

    for hypothesis_id, hypothesis in hypotheses_by_id.items():

        evidence_by_hypothesis[hypothesis_id] = {
            "hypothesis": hypothesis.to_dict(),
            "tests": [],
        }

    for result in evidence_results:

        if result.hypothesis_id not in evidence_by_hypothesis:
            continue

        test = tests_by_id.get(result.test_id)

        compact_result = {
            "test_id": result.test_id,
            "objective": (
                test.objective
                if test is not None
                else None
            ),
            "capability": result.capability,
            "execution_status": result.status,
            "evidence_status": result.evidence.get(
                "evidence_status"
            ),
            "observations": [],
        }

        # --------------------------------------------
        # Reachability evidence
        # --------------------------------------------

        if result.capability == "reachability":

            reachability = result.evidence.get(
                "reachability_evidence",
                {},
            )

            compact_result["observations"] = (
                reachability.get(
                    "observations",
                    [],
                )
            )

            path = result.evidence.get(
                "path_evidence",
                {},
            )

            compact_result["structural_path"] = (
                path.get("shortest_path")
            )

        # --------------------------------------------
        # Network-state evidence
        # --------------------------------------------

        elif result.capability == "network_state":

            network_state = result.evidence.get(
                "network_state_evidence",
                {},
            )

            compact_result["observations"] = (
                network_state.get(
                    "observations",
                    [],
                )
            )

        # --------------------------------------------
        # Topology evidence
        # --------------------------------------------

        elif result.capability == "topology":

            compact_result["topology_evidence"] = (
                result.evidence
            )

        evidence_by_hypothesis[
            result.hypothesis_id
        ]["tests"].append(
            compact_result
        )

    return {
        "hypotheses": evidence_by_hypothesis
    }

##ROOT CAUSE & REMEDIATION — DOMAIN OBJECTS

In [32]:
# ============================================================
# ROOT CAUSE & REMEDIATION — DOMAIN OBJECTS
# ============================================================

@dataclass
class RemediationStep:
    """One concrete, ordered remediation action."""

    step_number: int
    action: str
    rationale: str
    risk_level: str  # "low" | "medium" | "high"

    def validate(self) -> None:
        if self.step_number < 1:
            raise ValueError("step_number must be >= 1.")
        if not self.action.strip():
            raise ValueError("action cannot be empty.")
        if self.risk_level not in {"low", "medium", "high"}:
            raise ValueError(
                f"risk_level must be low/medium/high, got '{self.risk_level}'."
            )

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass
class RootCauseRecommendation:
    """
    Final output of an investigation: either a committed root cause
    with remediation, or an explicit statement that evidence was
    insufficient to commit to one.
    """

    incident_id: str
    status: str  # "root_cause_identified" | "insufficient_evidence"

    hypothesis_id: Optional[str] = None
    root_cause_explanation: Optional[str] = None
    confidence: Optional[float] = None

    remediation_steps: List[RemediationStep] = field(default_factory=list)
    residual_risk: Optional[str] = None

    challenge_acknowledged: Optional[str] = None  # how the challenger's dispute (if any) was addressed

    def validate(self) -> None:
        valid_statuses = {"root_cause_identified", "insufficient_evidence"}

        if self.status not in valid_statuses:
            raise ValueError(
                f"status must be one of {sorted(valid_statuses)}, "
                f"got '{self.status}'."
            )

        if self.status == "root_cause_identified":
            if not self.hypothesis_id:
                raise ValueError(
                    "hypothesis_id is required when status is "
                    "'root_cause_identified'."
                )
            if not self.root_cause_explanation:
                raise ValueError(
                    "root_cause_explanation is required when status "
                    "is 'root_cause_identified'."
                )
            if not self.remediation_steps:
                raise ValueError(
                    "At least one remediation step is required when "
                    "status is 'root_cause_identified'."
                )
            for step in self.remediation_steps:
                step.validate()

    def to_dict(self) -> Dict[str, Any]:
        return {
            "incident_id": self.incident_id,
            "status": self.status,
            "hypothesis_id": self.hypothesis_id,
            "root_cause_explanation": self.root_cause_explanation,
            "confidence": self.confidence,
            "remediation_steps": [s.to_dict() for s in self.remediation_steps],
            "residual_risk": self.residual_risk,
            "challenge_acknowledged": self.challenge_acknowledged,
        }

#6. EVIDENCE GRAPH

Agents reason over this graph. Executors populate and query it.

In [33]:
class EvidenceGraph:
    """
    Domain wrapper around a NetworkX MultiDiGraph.

    Agents and executors interact with this domain class rather than
    accessing NetworkX directly.
    """

    def __init__(self) -> None:
        self._graph = nx.MultiDiGraph()

        self._entities: Dict[str, NetworkEntity] = {}
        self._relationships: Dict[str, NetworkRelationship] = {}
        self._observations: Dict[str, ObservationRecord] = {}
        self._events: Dict[str, EventRecord] = {}
        self._changes: Dict[str, ChangeRecord] = {}

    # --------------------------------------------------------
    # Counts and summaries
    # --------------------------------------------------------

    @property
    def entity_count(self) -> int:
        return len(self._entities)

    @property
    def relationship_count(self) -> int:
        return len(self._relationships)

    @property
    def observation_count(self) -> int:
        return len(self._observations)

    @property
    def event_count(self) -> int:
        return len(self._events)

    @property
    def change_count(self) -> int:
        return len(self._changes)

    def summary(self) -> Dict[str, int]:
        """Return basic Evidence Graph record counts."""

        return {
            "entities": self.entity_count,
            "relationships": self.relationship_count,
            "observations": self.observation_count,
            "events": self.event_count,
            "changes": self.change_count,
        }

    # --------------------------------------------------------
    # Entity and relationship mutation
    # --------------------------------------------------------

    def add_entity(self, entity: NetworkEntity) -> None:
        entity.validate()

        if entity.entity_id in self._entities:
            raise ValueError(
                f"Entity '{entity.entity_id}' already exists."
            )

        self._entities[entity.entity_id] = entity

        self._graph.add_node(
            entity.entity_id,
            entity_type=entity.entity_type.value,
            name=entity.name,
            attributes=dict(entity.attributes),
            source_ids=list(entity.source_ids),
        )

    def upsert_entity(self, entity: NetworkEntity) -> None:
        entity.validate()

        self._entities[entity.entity_id] = entity

        self._graph.add_node(
            entity.entity_id,
            entity_type=entity.entity_type.value,
            name=entity.name,
            attributes=dict(entity.attributes),
            source_ids=list(entity.source_ids),
        )

    def add_relationship(
        self,
        relationship: NetworkRelationship,
    ) -> None:
        relationship.validate()

        if relationship.relationship_id in self._relationships:
            raise ValueError(
                f"Relationship "
                f"'{relationship.relationship_id}' already exists."
            )

        if relationship.source_entity_id not in self._entities:
            raise KeyError(
                f"Unknown source entity: "
                f"{relationship.source_entity_id}"
            )

        if relationship.target_entity_id not in self._entities:
            raise KeyError(
                f"Unknown target entity: "
                f"{relationship.target_entity_id}"
            )

        self._relationships[
            relationship.relationship_id
        ] = relationship

        self._graph.add_edge(
            relationship.source_entity_id,
            relationship.target_entity_id,
            key=relationship.relationship_id,
            relationship_id=relationship.relationship_id,
            relationship_type=relationship.relationship_type.value,
            attributes=dict(relationship.attributes),
            valid_from=relationship.valid_from,
            valid_to=relationship.valid_to,
            source_ids=list(relationship.source_ids),
        )

    # --------------------------------------------------------
    # Evidence mutation
    # --------------------------------------------------------

    def add_observation(
        self,
        observation: ObservationRecord,
    ) -> None:
        observation.validate()

        if observation.observation_id in self._observations:
            raise ValueError(
                f"Observation "
                f"'{observation.observation_id}' already exists."
            )

        if observation.entity_id not in self._entities:
            raise KeyError(
                f"Unknown observation entity: "
                f"{observation.entity_id}"
            )

        self._observations[
            observation.observation_id
        ] = observation

    def add_event(self, event: EventRecord) -> None:
        event.validate()

        if event.event_id in self._events:
            raise ValueError(
                f"Event '{event.event_id}' already exists."
            )

        missing_entities = [
            entity_id
            for entity_id in event.entity_ids
            if entity_id not in self._entities
        ]

        if missing_entities:
            raise KeyError(
                f"Unknown event entities: {missing_entities}"
            )

        self._events[event.event_id] = event

    def add_change(self, change: ChangeRecord) -> None:
        change.validate()

        if change.change_id in self._changes:
            raise ValueError(
                f"Change '{change.change_id}' already exists."
            )

        missing_entities = [
            entity_id
            for entity_id in change.entity_ids
            if entity_id not in self._entities
        ]

        if missing_entities:
            raise KeyError(
                f"Unknown change entities: {missing_entities}"
            )

        self._changes[change.change_id] = change

    # --------------------------------------------------------
    # Entity and relationship queries
    # --------------------------------------------------------

    def get_entity(
        self,
        entity_id: str,
    ) -> NetworkEntity:
        try:
            return self._entities[entity_id]
        except KeyError as exc:
            raise KeyError(
                f"Unknown entity: {entity_id}"
            ) from exc

    def get_relationship(
        self,
        relationship_id: str,
    ) -> NetworkRelationship:
        try:
            return self._relationships[relationship_id]
        except KeyError as exc:
            raise KeyError(
                f"Unknown relationship: {relationship_id}"
            ) from exc

    def list_entities(
        self,
        entity_type: Optional[EntityType] = None,
    ) -> List[NetworkEntity]:
        entities = list(self._entities.values())

        if entity_type is None:
            return entities

        return [
            entity
            for entity in entities
            if entity.entity_type == entity_type
        ]

    def get_neighbors(
        self,
        entity_id: str,
        relationship_type: Optional[
            RelationshipType
        ] = None,
    ) -> List[NetworkEntity]:
        if entity_id not in self._entities:
            raise KeyError(f"Unknown entity: {entity_id}")

        neighbor_ids = set()

        for _, target_id, _, edge_data in self._graph.out_edges(
            entity_id,
            keys=True,
            data=True,
        ):
            if (
                relationship_type is None
                or edge_data.get("relationship_type")
                == relationship_type.value
            ):
                neighbor_ids.add(target_id)

        for source_id, _, _, edge_data in self._graph.in_edges(
            entity_id,
            keys=True,
            data=True,
        ):
            if (
                relationship_type is None
                or edge_data.get("relationship_type")
                == relationship_type.value
            ):
                neighbor_ids.add(source_id)

        return [
            self._entities[neighbor_id]
            for neighbor_id in sorted(neighbor_ids)
        ]

    # --------------------------------------------------------
    # Topology queries
    # --------------------------------------------------------

    def _connectivity_graph(self) -> nx.DiGraph:
      """
      Return a directed connectivity projection of the Evidence Graph.

      Relationships marked as bidirectional are represented in both
      directions for path analysis.
      """

      graph = nx.DiGraph()

      graph.add_nodes_from(self._graph.nodes(data=True))

      for (
        source_id,
        target_id,
        _,
        edge_data,
      ) in self._graph.edges(
        keys=True,
        data=True,
      ):

          graph.add_edge(
              source_id,
              target_id,
              **edge_data,
          )

          relationship_attributes = edge_data.get("attributes",
            {},)

          if relationship_attributes.get(
            "bidirectional",
            False,
          ):
              graph.add_edge(
                  target_id,
                  source_id,
                  **edge_data,)

      return graph

    def find_paths(
        self,
        source_entity_id: str,
        target_entity_id: str,
        cutoff: Optional[int] = None,
    ) -> List[List[str]]:
        if source_entity_id not in self._entities:
            raise KeyError(
                f"Unknown source entity: {source_entity_id}"
            )

        if target_entity_id not in self._entities:
            raise KeyError(
                f"Unknown target entity: {target_entity_id}"
            )

        return list(
            nx.all_simple_paths(
                self._connectivity_graph(),
                source=source_entity_id,
                target=target_entity_id,
                cutoff=cutoff,
            )
        )

    def shortest_path(
        self,
        source_entity_id: str,
        target_entity_id: str,
    ) -> List[str]:
        if source_entity_id not in self._entities:
            raise KeyError(
                f"Unknown source entity: {source_entity_id}"
            )

        if target_entity_id not in self._entities:
            raise KeyError(
                f"Unknown target entity: {target_entity_id}"
            )

        try:
            return nx.shortest_path(
                self._connectivity_graph(),
                source=source_entity_id,
                target=target_entity_id,
            )
        except nx.NetworkXNoPath as exc:
            raise ValueError(
                f"No directed path exists between "
                f"'{source_entity_id}' and "
                f"'{target_entity_id}'."
            ) from exc

    def descendants(self,
      entity_id: str,
    ) -> List[str]:
      """
      Return entities downstream according to explicitly directed
      Evidence Graph relationships.

      Bidirectional connectivity semantics are intentionally not
      applied here.
      """

      if entity_id not in self._entities:
        raise KeyError(f"Unknown entity: {entity_id}")

      simple_graph = nx.DiGraph(self._graph)

      return sorted(
        nx.descendants(
            simple_graph,
            entity_id,)
    )

    # --------------------------------------------------------
    # Evidence queries
    # --------------------------------------------------------

    def get_observations(
        self,
        entity_ids: Optional[List[str]] = None,
        metric_names: Optional[List[str]] = None,
        start_time: Optional[str] = None,
        end_time: Optional[str] = None,
    ) -> List[ObservationRecord]:
        results = list(self._observations.values())

        if entity_ids is not None:
            entity_id_set = set(entity_ids)
            results = [
                record
                for record in results
                if record.entity_id in entity_id_set
            ]

        if metric_names is not None:
            metric_name_set = set(metric_names)
            results = [
                record
                for record in results
                if record.metric_name in metric_name_set
            ]

        if start_time is not None:
            results = [
                record
                for record in results
                if record.observed_at >= start_time
            ]

        if end_time is not None:
            results = [
                record
                for record in results
                if record.observed_at <= end_time
            ]

        return sorted(
            results,
            key=lambda record: record.observed_at,
        )


    def get_events(
        self,
        entity_ids: Optional[List[str]] = None,
        event_types: Optional[List[str]] = None,
        start_time: Optional[str] = None,
        end_time: Optional[str] = None,
    ) -> List[EventRecord]:
        results = list(self._events.values())

        if entity_ids is not None:
            entity_id_set = set(entity_ids)
            results = [
                record
                for record in results
                if entity_id_set & set(record.entity_ids)
            ]

        if event_types is not None:
            event_type_set = set(event_types)
            results = [
                record
                for record in results
                if record.event_type in event_type_set
            ]

        if start_time is not None:
            results = [
                record
                for record in results
                if record.occurred_at >= start_time
            ]

        if end_time is not None:
            results = [
                record
                for record in results
                if record.occurred_at <= end_time
            ]

        return sorted(
            results,
            key=lambda record: record.occurred_at,
        )

#7. EVIDENCE ADAPTER LAYER




In [34]:
@dataclass
class EvidenceAdapterResult:
    """
    Standardized output returned by every evidence adapter.

    An adapter may populate some or all record collections depending
    on the source. For example, Topology Zoo may return entities and
    relationships, while Prometheus may primarily return observations.
    """

    source_id: str

    entities: List[NetworkEntity] = field(default_factory=list)
    relationships: List[NetworkRelationship] = field(default_factory=list)
    observations: List[ObservationRecord] = field(default_factory=list)
    events: List[EventRecord] = field(default_factory=list)
    changes: List[ChangeRecord] = field(default_factory=list)

    metadata: Dict[str, Any] = field(default_factory=dict)
    warnings: List[str] = field(default_factory=list)

    def validate(self) -> None:
        if not self.source_id.strip():
            raise ValueError("source_id cannot be empty.")

        for entity in self.entities:
            entity.validate()

        for relationship in self.relationships:
            relationship.validate()

        for observation in self.observations:
            observation.validate()

        for event in self.events:
            event.validate()

        for change in self.changes:
            change.validate()

    def summary(self) -> Dict[str, int]:
        return {
            "entities": len(self.entities),
            "relationships": len(self.relationships),
            "observations": len(self.observations),
            "events": len(self.events),
            "changes": len(self.changes),
            "warnings": len(self.warnings),
        }


class EvidenceAdapter(ABC):
    """
    Abstract translator between a raw evidence source and the
    vendor-neutral Evidence Graph domain model.

    Concrete adapters must:
      1. read a source;
      2. normalize source-specific schemas;
      3. create typed domain records; and
      4. return an EvidenceAdapterResult.

    Adapters do not diagnose incidents or evaluate hypotheses.
    """

    def __init__(
        self,
        source_config: EvidenceSourceConfig,
    ) -> None:
        self.source_config = source_config
        self.source_config.validate()

    @property
    def source_id(self) -> str:
        return self.source_config.source_id

    @abstractmethod
    def load(self) -> EvidenceAdapterResult:
        """
        Load and normalize the configured evidence source.

        Returns
        -------
        EvidenceAdapterResult
            Vendor-neutral entities, relationships, observations,
            events, and changes.
        """
        raise NotImplementedError

    def validate_result(
        self,
        result: EvidenceAdapterResult,
    ) -> None:
        """
        Validate the adapter output and ensure source attribution
        matches the adapter configuration.
        """

        result.validate()

        if result.source_id != self.source_id:
            raise ValueError(
                "Adapter result source_id does not match the "
                f"configured source_id: expected '{self.source_id}', "
                f"received '{result.source_id}'."
            )

In [35]:
def ingest_adapter_result_executor(
    evidence_graph: EvidenceGraph,
    adapter_result: EvidenceAdapterResult,
    *,
    upsert_entities: bool = True,
) -> Dict[str, Any]:
    """
    Insert normalized adapter records into an EvidenceGraph.

    This is an executor because ingestion is deterministic. It does
    not interpret evidence or make root-cause conclusions.
    """

    adapter_result.validate()

    ingested_counts = {
        "entities": 0,
        "relationships": 0,
        "observations": 0,
        "events": 0,
        "changes": 0,
    }

    # Entities must be inserted first because every other record may
    # reference them.
    for entity in adapter_result.entities:
        if upsert_entities:
            evidence_graph.upsert_entity(entity)
        else:
            evidence_graph.add_entity(entity)

        ingested_counts["entities"] += 1

    # Relationships require both endpoint entities to already exist.
    for relationship in adapter_result.relationships:
        evidence_graph.add_relationship(relationship)
        ingested_counts["relationships"] += 1

    for observation in adapter_result.observations:
        evidence_graph.add_observation(observation)
        ingested_counts["observations"] += 1

    for event in adapter_result.events:
        evidence_graph.add_event(event)
        ingested_counts["events"] += 1

    for change in adapter_result.changes:
        evidence_graph.add_change(change)
        ingested_counts["changes"] += 1

    return {
        "executor": "ingest_adapter_result_executor",
        "source_id": adapter_result.source_id,
        "ingested_counts": ingested_counts,
        "warnings": list(adapter_result.warnings),
        "adapter_metadata": dict(adapter_result.metadata),
        "graph_summary": evidence_graph.summary(),
    }

##Adapter Registry

In [36]:
# ============================================================
# ADAPTER REGISTRY
# ============================================================

ADAPTER_REGISTRY: Dict[str, type[EvidenceAdapter]] = {}


def register_adapter(
    adapter_name: str,
    adapter_class: type[EvidenceAdapter],
) -> None:
    """
    Register an EvidenceAdapter implementation by name.
    """

    if not adapter_name.strip():
        raise ValueError("adapter_name cannot be empty.")

    if not issubclass(adapter_class, EvidenceAdapter):
        raise TypeError(
            f"{adapter_class.__name__} must inherit from EvidenceAdapter."
        )

    if adapter_name in ADAPTER_REGISTRY:
        raise ValueError(
            f"Adapter '{adapter_name}' is already registered."
        )

    ADAPTER_REGISTRY[adapter_name] = adapter_class


def get_adapter_class(
    adapter_name: str,
) -> type[EvidenceAdapter]:
    """
    Resolve the adapter class referenced by EvidenceSourceConfig.
    """

    try:
        return ADAPTER_REGISTRY[adapter_name]

    except KeyError as exc:
        raise KeyError(
            f"No adapter registered as '{adapter_name}'. "
            f"Available adapters: {sorted(ADAPTER_REGISTRY)}"
        ) from exc

##Scenario-Specific NIKA Adapter

In [37]:
class NIKASimpleBGPOperationalAdapter(EvidenceAdapter):
    """
    Normalize FIRST-RESPONSE operational evidence for the NIKA
    simple_bgp link_down scenario into generic ObservationRecord
    objects.

    This represents what would realistically be available in an
    initial telemetry snapshot: interface operational status and
    basic reachability. It deliberately does NOT include BGP
    session state or deep hardware diagnostics — those require a
    separate, explicit diagnostic query (see
    DeepDiagnosticsCapability), mirroring how a real first alert
    carries limited signal compared to what's available on request.

    Version 1 is a deterministic fixture representing the known
    injected incident. A later version will obtain the same evidence
    from NIKA's live MCP/telemetry interfaces.
    """

    def load(self) -> EvidenceAdapterResult:
        scenario_id = self.source_config.metadata.get("scenario_id")
        problem_id = self.source_config.metadata.get("problem_id")

        if scenario_id != "simple_bgp":
            raise ValueError(
                "NIKASimpleBGPOperationalAdapter only supports "
                "scenario_id='simple_bgp'."
            )

        if problem_id != "link_down":
            raise ValueError(
                "This initial operational adapter only supports "
                "problem_id='link_down'."
            )

        observed_at = self.source_config.metadata.get(
            "observed_at",
            "2026-08-07T00:00:01Z",
        )

        observations = [
            # ------------------------------------------------
            # Inter-router interface state (first response only —
            # BGP session state is a tier-2 deep diagnostic)
            # ------------------------------------------------

            ObservationRecord(
                observation_id=(
                    "nika:simple_bgp:obs:"
                    "router1-router2-interface-state"
                ),
                source_id=self.source_id,
                entity_id="nika:simple_bgp:router1",
                metric_name="interface_oper_status",
                metric_value="down",
                observed_at=observed_at,
                dimensions={
                    "peer_entity_id":
                        "nika:simple_bgp:router2",
                },
            ),

            ObservationRecord(
                observation_id=(
                    "nika:simple_bgp:obs:"
                    "router2-router1-interface-state"
                ),
                source_id=self.source_id,
                entity_id="nika:simple_bgp:router2",
                metric_name="interface_oper_status",
                metric_value="down",
                observed_at=observed_at,
                dimensions={
                    "peer_entity_id":
                        "nika:simple_bgp:router1",
                },
            ),

            # ------------------------------------------------
            # Inter-router reachability
            # ------------------------------------------------

            ObservationRecord(
                observation_id=(
                    "nika:simple_bgp:obs:"
                    "router1-router2-ping"
                ),
                source_id=self.source_id,
                entity_id="nika:simple_bgp:router1",
                metric_name="ping_success",
                metric_value=False,
                observed_at=observed_at,
                dimensions={
                    "target_entity_id":
                        "nika:simple_bgp:router2",
                },
            ),

            # ------------------------------------------------
            # Healthy edge reachability
            # ------------------------------------------------

            ObservationRecord(
                observation_id=(
                    "nika:simple_bgp:obs:"
                    "pc1-router1-ping"
                ),
                source_id=self.source_id,
                entity_id="nika:simple_bgp:pc1",
                metric_name="ping_success",
                metric_value=True,
                observed_at=observed_at,
                dimensions={
                    "target_entity_id":
                        "nika:simple_bgp:router1",
                },
            ),

            ObservationRecord(
                observation_id=(
                    "nika:simple_bgp:obs:"
                    "router2-pc2-ping"
                ),
                source_id=self.source_id,
                entity_id="nika:simple_bgp:router2",
                metric_name="ping_success",
                metric_value=True,
                observed_at=observed_at,
                dimensions={
                    "target_entity_id":
                        "nika:simple_bgp:pc2",
                },
            ),
        ]

        result = EvidenceAdapterResult(
            source_id=self.source_id,
            observations=observations,
            metadata={
                "provider": "nika",
                "scenario_id": scenario_id,
                "problem_id": problem_id,
                "adapter": self.__class__.__name__,
                "integration_stage": "operational_evidence_fixture",
                "evidence_tier": "first_response",
            },
        )

        self.validate_result(result)

        return result

##NIKA SimpleBGP Operational Adapter

Registering this adapter

In [38]:
register_adapter(
    "NIKASimpleBGPOperationalAdapter",
    NIKASimpleBGPOperationalAdapter,
)

##NIKA Simple BGP Adapter

In [39]:
class NIKASimpleBGPAdapter(EvidenceAdapter):
    """
    Normalize NIKA's simple_bgp scenario into Evidence Graph
    domain records.

    Initial topology:

        pc1 -- router1 -- router2 -- pc2

    This adapter is intentionally narrow. It validates the NIKA
    integration path before we generalize into a reusable NIKAAdapter.
    """

    def load(self) -> EvidenceAdapterResult:
        scenario_id = self.source_config.metadata.get("scenario_id")

        if scenario_id != "simple_bgp":
            raise ValueError(
                "NIKASimpleBGPAdapter only supports "
                "scenario_id='simple_bgp'."
            )

        entities = [
            NetworkEntity(
                entity_id="nika:simple_bgp:pc1",
                entity_type=EntityType.HOST,
                name="pc1",
                attributes={
                    "provider": "nika",
                    "scenario": "simple_bgp",
                    "role": "host",
                },
                source_ids=[self.source_id],
            ),

            NetworkEntity(
                entity_id="nika:simple_bgp:router1",
                entity_type=EntityType.DEVICE,
                name="router1",
                attributes={
                    "provider": "nika",
                    "scenario": "simple_bgp",
                    "role": "bgp_router",
                },
                source_ids=[self.source_id],
            ),

            NetworkEntity(
                entity_id="nika:simple_bgp:router2",
                entity_type=EntityType.DEVICE,
                name="router2",
                attributes={
                    "provider": "nika",
                    "scenario": "simple_bgp",
                    "role": "bgp_router",
                },
                source_ids=[self.source_id],
            ),

            NetworkEntity(
                entity_id="nika:simple_bgp:pc2",
                entity_type=EntityType.HOST,
                name="pc2",
                attributes={
                    "provider": "nika",
                    "scenario": "simple_bgp",
                    "role": "host",
                },
                source_ids=[self.source_id],
            ),
        ]

        relationships = [
            NetworkRelationship(
                relationship_id="nika:simple_bgp:pc1-router1",
                source_entity_id="nika:simple_bgp:pc1",
                target_entity_id="nika:simple_bgp:router1",
                relationship_type=RelationshipType.CONNECTED_TO,
                attributes={
                    "bidirectional": True,
                },
                source_ids=[self.source_id],
            ),

            NetworkRelationship(
                relationship_id="nika:simple_bgp:router1-router2",
                source_entity_id="nika:simple_bgp:router1",
                target_entity_id="nika:simple_bgp:router2",
                relationship_type=RelationshipType.CONNECTED_TO,
                attributes={
                    "bidirectional": True,
                    "routing_protocol": "bgp",
                },
                source_ids=[self.source_id],
            ),

            NetworkRelationship(
                relationship_id="nika:simple_bgp:router2-pc2",
                source_entity_id="nika:simple_bgp:router2",
                target_entity_id="nika:simple_bgp:pc2",
                relationship_type=RelationshipType.CONNECTED_TO,
                attributes={
                    "bidirectional": True,
                },
                source_ids=[self.source_id],
            ),
        ]

        result = EvidenceAdapterResult(
            source_id=self.source_id,
            entities=entities,
            relationships=relationships,
            metadata={
                "provider": "nika",
                "scenario_id": "simple_bgp",
                "adapter": self.__class__.__name__,
                "integration_stage": "static_topology",
            },
        )

        self.validate_result(result)

        return result

Registering this adapter

In [40]:
register_adapter(
    "NIKASimpleBGPAdapter",
    NIKASimpleBGPAdapter,
)

##Pager Duty Evidence Adapter

In [41]:
class PagerDutySimpleBGPIncidentAdapter(EvidenceAdapter):
    """
    Normalize a PagerDuty incident (shaped like a real PagerDuty
    incident.trigger webhook payload) for the NIKA simple_bgp
    link_down scenario into a generic EventRecord.

    Version 1 is a deterministic fixture representing what a
    monitoring vendor would have actually sent: a symptom-level
    alert with no root cause, referencing a service name rather
    than an internal entity ID. Entity resolution from the vendor's
    service name to internal entity IDs happens here, at the
    adapter boundary — reasoning agents never see the raw
    PagerDuty payload or vendor-specific naming.

    A later version will consume this from PagerDuty's real
    Events API / webhook delivery.
    """

    SERVICE_NAME_TO_ENTITY_IDS = {
        "core-network-router1-router2-link": [
            "nika:simple_bgp:router1",
            "nika:simple_bgp:router2",
        ],
    }

    def load(self) -> EvidenceAdapterResult:
        scenario_id = self.source_config.metadata.get("scenario_id")
        problem_id = self.source_config.metadata.get("problem_id")

        if scenario_id != "simple_bgp":
            raise ValueError(
                "PagerDutySimpleBGPIncidentAdapter only supports "
                "scenario_id='simple_bgp'."
            )

        if problem_id != "link_down":
            raise ValueError(
                "This initial PagerDuty adapter only supports "
                "problem_id='link_down'."
            )

        raw_incident = {
            "id": "PD1234567",
            "type": "incident",
            "status": "triggered",
            "title": (
                "High packet loss / connection timeout on "
                "core-network-router1-router2-link"
            ),
            "urgency": "high",
            "service": {
                "id": "PSVC001",
                "summary": "core-network-router1-router2-link",
            },
            "created_at": "2026-08-07T00:00:05Z",
            "html_url": "https://nika.pagerduty.com/incidents/PD1234567",
            "assignments": [
                {"assignee": {"summary": "NetOps On-Call"}}
            ],
            "custom_details": {
                "monitor": "Synthetic BGP Peer Reachability Check",
                "alert_source": "nika-synthetic-monitoring",
                "description": (
                    "Synthetic monitor detected 100% packet loss "
                    "between BGP peers on the core interconnect. "
                    "No further diagnosis available — see network "
                    "team for root cause."
                ),
            },
        }

        service_name = raw_incident["service"]["summary"]
        entity_ids = self.SERVICE_NAME_TO_ENTITY_IDS.get(service_name, [])

        if not entity_ids:
            raise ValueError(
                "PagerDutySimpleBGPIncidentAdapter could not resolve "
                f"service '{service_name}' to any known entity IDs."
            )

        event = EventRecord(
            event_id=f"pagerduty:{raw_incident['id']}",
            source_id=self.source_id,
            event_type="vendor_incident",
            occurred_at=raw_incident["created_at"],
            entity_ids=entity_ids,
            severity=raw_incident["urgency"],
            message=raw_incident["title"],
            attributes={
                "vendor": "pagerduty",
                "vendor_incident_id": raw_incident["id"],
                "status": raw_incident["status"],
                "url": raw_incident["html_url"],
                "assigned_to": raw_incident["assignments"][0]["assignee"]["summary"],
                "monitor": raw_incident["custom_details"]["monitor"],
                "vendor_description": raw_incident["custom_details"]["description"],
            },
        )

        result = EvidenceAdapterResult(
            source_id=self.source_id,
            events=[event],
            metadata={
                "provider": "pagerduty",
                "scenario_id": scenario_id,
                "problem_id": problem_id,
                "adapter": self.__class__.__name__,
                "integration_stage": "vendor_alert_fixture",
            },
        )

        self.validate_result(result)

        return result


register_adapter("PagerDutySimpleBGPIncidentAdapter", PagerDutySimpleBGPIncidentAdapter)

#8. DETERMINISTIC EXECUTORS

In [42]:
def entity_lookup_executor(
    evidence_graph: EvidenceGraph,
    entity_id: str,
) -> Dict[str, Any]:
    entity = evidence_graph.get_entity(entity_id)

    return {
        "executor": "entity_lookup_executor",
        "entity": entity.to_dict(),
        "neighbors": [
            neighbor.to_dict()
            for neighbor in evidence_graph.get_neighbors(
                entity_id
            )
        ],
    }


def path_analysis_executor(
    evidence_graph: EvidenceGraph,
    source_entity_id: str,
    target_entity_id: str,
    *,
    include_all_paths: bool = False,
    cutoff: Optional[int] = None,
) -> Dict[str, Any]:
    shortest_path = evidence_graph.shortest_path(
        source_entity_id,
        target_entity_id,
    )

    result = {
        "executor": "path_analysis_executor",
        "source_entity_id": source_entity_id,
        "target_entity_id": target_entity_id,
        "shortest_path": shortest_path,
        "hop_count": len(shortest_path) - 1,
    }

    if include_all_paths:
        all_paths = evidence_graph.find_paths(
            source_entity_id,
            target_entity_id,
            cutoff=cutoff,
        )

        result["all_paths"] = all_paths
        result["path_count"] = len(all_paths)

    return result


def blast_radius_executor(
    evidence_graph: EvidenceGraph,
    failed_entity_id: str,
) -> Dict[str, Any]:
    failed_entity = evidence_graph.get_entity(
        failed_entity_id
    )

    simple_graph = nx.DiGraph(evidence_graph._graph)

    affected_entity_ids = sorted(
        nx.descendants(
            simple_graph,
            failed_entity_id,
        )
    )

    return {
        "executor": "blast_radius_executor",
        "failed_entity": failed_entity.to_dict(),
        "affected_entity_ids": affected_entity_ids,
        "affected_count": len(affected_entity_ids),
    }

In [43]:
test_graph = EvidenceGraph()

test_entities = [
    NetworkEntity(
        entity_id="site-sfo",
        entity_type=EntityType.SITE,
        name="San Francisco Site",
    ),
    NetworkEntity(
        entity_id="host-client-01",
        entity_type=EntityType.HOST,
        name="Client 01",
    ),
    NetworkEntity(
        entity_id="router-r1",
        entity_type=EntityType.DEVICE,
        name="Router R1",
    ),
    NetworkEntity(
        entity_id="router-r2",
        entity_type=EntityType.DEVICE,
        name="Router R2",
    ),
    NetworkEntity(
        entity_id="service-app-01",
        entity_type=EntityType.SERVICE,
        name="Destination Service",
    ),
]

for entity in test_entities:
    test_graph.add_entity(entity)

test_relationships = [
    NetworkRelationship(
        relationship_id="rel-site-client",
        source_entity_id="site-sfo",
        target_entity_id="host-client-01",
        relationship_type=RelationshipType.CONTAINS,
    ),
    NetworkRelationship(
        relationship_id="rel-client-r1",
        source_entity_id="host-client-01",
        target_entity_id="router-r1",
        relationship_type=RelationshipType.CONNECTED_TO,
    ),
    NetworkRelationship(
        relationship_id="rel-r1-r2",
        source_entity_id="router-r1",
        target_entity_id="router-r2",
        relationship_type=RelationshipType.CONNECTED_TO,
    ),
    NetworkRelationship(
        relationship_id="rel-r2-service",
        source_entity_id="router-r2",
        target_entity_id="service-app-01",
        relationship_type=RelationshipType.CONNECTED_TO,
    ),
]

for relationship in test_relationships:
    test_graph.add_relationship(relationship)

test_graph.add_observation(
    ObservationRecord(
        observation_id="obs-r1-utilization",
        source_id="test_telemetry",
        entity_id="router-r1",
        metric_name="cpu_utilization",
        metric_value=42.5,
        unit="percent",
        observed_at="2026-08-04T14:00:00Z",
    )
)

print(test_graph.summary())

print(
    path_analysis_executor(
        evidence_graph=test_graph,
        source_entity_id="host-client-01",
        target_entity_id="service-app-01",
        include_all_paths=True,
    )
)

{'entities': 5, 'relationships': 4, 'observations': 1, 'events': 0, 'changes': 0}
{'executor': 'path_analysis_executor', 'source_entity_id': 'host-client-01', 'target_entity_id': 'service-app-01', 'shortest_path': ['host-client-01', 'router-r1', 'router-r2', 'service-app-01'], 'hop_count': 3, 'all_paths': [['host-client-01', 'router-r1', 'router-r2', 'service-app-01']], 'path_count': 1}


In [44]:
pprint.pprint(
    entity_lookup_executor(
        evidence_graph=test_graph,
        entity_id="router-r1",
    )
)

{'entity': {'attributes': {},
            'entity_id': 'router-r1',
            'entity_type': 'device',
            'name': 'Router R1',
            'source_ids': []},
 'executor': 'entity_lookup_executor',
 'neighbors': [{'attributes': {},
                'entity_id': 'host-client-01',
                'entity_type': 'host',
                'name': 'Client 01',
                'source_ids': []},
               {'attributes': {},
                'entity_id': 'router-r2',
                'entity_type': 'device',
                'name': 'Router R2',
                'source_ids': []}]}


In [45]:
print(
    blast_radius_executor(
        evidence_graph=test_graph,
        failed_entity_id="router-r1",
    )
)

{'executor': 'blast_radius_executor', 'failed_entity': {'entity_id': 'router-r1', 'entity_type': 'device', 'name': 'Router R1', 'attributes': {}, 'source_ids': []}, 'affected_entity_ids': ['router-r2', 'service-app-01'], 'affected_count': 2}


In [46]:
def load_evidence_source_executor(
    source_config: EvidenceSourceConfig,
    evidence_graph: EvidenceGraph,
) -> Dict[str, Any]:
    """
    Resolve the configured adapter, normalize one source,
    and ingest the result into the Evidence Graph.
    """

    source_config.validate()

    adapter_class = get_adapter_class(
        source_config.adapter_name
    )

    adapter = adapter_class(
        source_config=source_config
    )

    adapter_result = adapter.load()

    adapter.validate_result(
        adapter_result
    )

    ingestion_result = ingest_adapter_result_executor(
        evidence_graph=evidence_graph,
        adapter_result=adapter_result,
    )

    return {
        "executor": "load_evidence_source_executor",
        "source_id": source_config.source_id,
        "adapter_name": source_config.adapter_name,
        "adapter_summary": adapter_result.summary(),
        "ingestion_result": ingestion_result,
    }

In [47]:
def load_investigation_evidence_executor(
    investigation_config: NetworkInvestigationConfig,
    evidence_graph: Optional[EvidenceGraph] = None,
) -> Dict[str, Any]:
    """
    Load every enabled evidence source configured for an investigation.

    Required-source failures stop execution.
    Optional-source failures are recorded and skipped.
    """

    investigation_config.validate()

    if evidence_graph is None:
        evidence_graph = EvidenceGraph()

    source_results: Dict[str, Any] = {}
    source_errors: Dict[str, str] = {}

    for source_config in investigation_config.enabled_sources:

        try:
            result = load_evidence_source_executor(
                source_config=source_config,
                evidence_graph=evidence_graph,
            )

            source_results[
                source_config.source_id
            ] = result

            print(
                f"✓ Loaded {source_config.source_id} "
                f"using {source_config.adapter_name}"
            )

        except Exception as exc:
            source_errors[
                source_config.source_id
            ] = str(exc)

            if source_config.required:
                raise RuntimeError(
                    "Required evidence source failed: "
                    f"{source_config.source_id}"
                ) from exc

            print(
                f"⚠ Optional evidence source unavailable: "
                f"{source_config.source_id}"
            )

    return {
        "executor": "load_investigation_evidence_executor",
        "investigation_id": (
            investigation_config.investigation_id
        ),
        "source_results": source_results,
        "source_errors": source_errors,
        "graph_summary": evidence_graph.summary(),
        "evidence_graph": evidence_graph,
    }

In [48]:
nika_simple_bgp_config = NetworkInvestigationConfig(
    investigation_id="nika-simple-bgp-link-down-001",
    investigation_name="NIKA Simple BGP Link Failure",
    environment_name="nika",
    investigation_type="connectivity_failure",

    scenario_provider="nika",
    scenario_id="simple_bgp",

    incident_seed=IncidentSeed(
        reported_symptom=(
            "Users report connectivity issues between "
            "the two endpoint hosts."
        ),
        incident_start_time="2026-08-07T00:00:00Z",
        reported_entities=[
            "nika:simple_bgp:pc1",
            "nika:simple_bgp:pc2",
        ],
        source="nika",
        initial_severity="high",
    ),

    twin_source_id="nika_simple_bgp_topology",

    evidence_sources=[
        EvidenceSourceConfig(
            source_id="nika_simple_bgp_topology",
            evidence_type=EvidenceType.TOPOLOGY,
            source_format=SourceFormat.API,
            access_mode=AccessMode.API,
            adapter_name="NIKASimpleBGPAdapter",
            connection_parameters={"provider": "nika",},
            required=True,
            metadata={
                "scenario_id": "simple_bgp",
                "problem_id": "link_down",
            },
        ),
        EvidenceSourceConfig(
        source_id="nika_simple_bgp_operational",
        evidence_type=EvidenceType.TELEMETRY,
        source_format=SourceFormat.API,
        access_mode=AccessMode.API,
        adapter_name="NIKASimpleBGPOperationalAdapter",
        connection_parameters={"provider": "nika",},
        required=True,
        metadata={
            "scenario_id": "simple_bgp",
            "problem_id": "link_down",
            "observed_at": "2026-08-07T00:00:01Z",
          },
        ),
        EvidenceSourceConfig(
            source_id="pagerduty_simple_bgp_incident",
            evidence_type=EvidenceType.EVENT,
            source_format=SourceFormat.API,
            access_mode=AccessMode.API,
            adapter_name="PagerDutySimpleBGPIncidentAdapter",
            connection_parameters={"provider": "pagerduty",},
            required=True,
            metadata={
                "scenario_id": "simple_bgp",
                "problem_id": "link_down",
            },
        ),
    ],

    investigation_policy=InvestigationPolicy(
        remediation_mode=RemediationMode.RECOMMEND_ONLY,
        require_human_approval=True,
        minimum_diagnosis_confidence=0.80,
    ),

    evaluation_config=EvaluationConfig(
        ground_truth_root_cause="link_down",
        ground_truth_fault_type="link_failure",
        ground_truth_entity_ids=[
            "nika:simple_bgp:router1",
            "nika:simple_bgp:router2",
        ],
    ),

    metadata={
        "nika_task": "simple_bgp_link_down",
        "integration_stage": "static_topology",
    },
)

In [49]:
nika_simple_bgp_config.validate()

nika_load_result = load_investigation_evidence_executor(
    investigation_config=nika_simple_bgp_config
)

nika_graph = nika_load_result["evidence_graph"]

print()
print("Graph summary:")
print(nika_graph.summary())

✓ Loaded nika_simple_bgp_topology using NIKASimpleBGPAdapter
✓ Loaded nika_simple_bgp_operational using NIKASimpleBGPOperationalAdapter
✓ Loaded pagerduty_simple_bgp_incident using PagerDutySimpleBGPIncidentAdapter

Graph summary:
{'entities': 4, 'relationships': 3, 'observations': 5, 'events': 1, 'changes': 0}


In [50]:
nika_path_result = path_analysis_executor(
    evidence_graph=nika_graph,
    source_entity_id="nika:simple_bgp:pc1",
    target_entity_id="nika:simple_bgp:pc2",
    include_all_paths=True,
)

print(nika_path_result)

{'executor': 'path_analysis_executor', 'source_entity_id': 'nika:simple_bgp:pc1', 'target_entity_id': 'nika:simple_bgp:pc2', 'shortest_path': ['nika:simple_bgp:pc1', 'nika:simple_bgp:router1', 'nika:simple_bgp:router2', 'nika:simple_bgp:pc2'], 'hop_count': 3, 'all_paths': [['nika:simple_bgp:pc1', 'nika:simple_bgp:router1', 'nika:simple_bgp:router2', 'nika:simple_bgp:pc2']], 'path_count': 1}


##Reachability Evidence Executor

In [51]:
def reachability_evidence_executor(
    evidence_graph: EvidenceGraph,
    source_entity_id: str,
    target_entity_id: str,
) -> Dict[str, Any]:
    """
    Retrieve normalized reachability observations between two entities.

    Expected future metrics may include:
      - ping_success
      - packet_loss
      - latency_ms
      - traceroute_hop_count

    The executor does not perform live network commands. It queries
    evidence already normalized into the Evidence Graph.
    """

    source_observations = evidence_graph.get_observations(
        entity_ids=[source_entity_id]
    )

    reachability_metrics = {
        "ping_success",
        "packet_loss",
        "latency_ms",
        "traceroute_hop_count",
    }

    matching_observations = []

    for observation in source_observations:

        if observation.metric_name not in reachability_metrics:
            continue

        observed_target = observation.dimensions.get(
            "target_entity_id"
        )

        if observed_target == target_entity_id:
            matching_observations.append(
                observation.to_dict()
            )

    return {
        "executor": "reachability_evidence_executor",
        "source_entity_id": source_entity_id,
        "target_entity_id": target_entity_id,
        "observations": matching_observations,
        "observation_count": len(
            matching_observations
        ),
    }

##Network State Evidence Executor

In [52]:
def network_state_evidence_executor(
    evidence_graph: EvidenceGraph,
    entity_ids: List[str],
) -> Dict[str, Any]:
    """
    Retrieve normalized operational-state observations for entities.

    Expected future metrics may include:
      - interface_oper_status
      - interface_admin_status
      - bgp_session_state
      - route_present
      - device_reachable
    """

    network_state_metrics = {
        "interface_oper_status",
        "interface_admin_status",
        "bgp_session_state",
        "route_present",
        "device_reachable",
    }

    observations = evidence_graph.get_observations(
        entity_ids=entity_ids
    )

    matching_observations = [
        observation.to_dict()
        for observation in observations
        if observation.metric_name
        in network_state_metrics
    ]

    return {
        "executor": "network_state_evidence_executor",
        "entity_ids": list(entity_ids),
        "observations": matching_observations,
        "observation_count": len(
            matching_observations
        ),
    }

##Evidence Collection Executor

In [53]:
@dataclass
class EvidenceTestResult:
    test_id: str
    hypothesis_id: str
    capability: str

    status: str
    evidence: Dict[str, Any] = field(default_factory=dict)
    error: Optional[str] = None

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

In [54]:
def execute_evidence_test(
    test: EvidenceTest,
    evidence_graph: EvidenceGraph,
    topology_capability: TopologyCapability,
    reachability_capability: ReachabilityCapability,
    network_state_capability: NetworkStateCapability,
    vendor_alert_capability: VendorAlertCapability,
    deep_diagnostics_capability: DeepDiagnosticsCapability,
) -> EvidenceTestResult:

    try:
        if test.capability == "topology":
            evidence = topology_capability.collect_evidence(
                evidence_graph=evidence_graph,
                **test.parameters,
            )

        elif test.capability == "reachability":
            evidence = reachability_capability.collect_evidence(
                evidence_graph=evidence_graph,
                **test.parameters,
            )

        elif test.capability == "network_state":
            evidence = network_state_capability.collect_evidence(
                evidence_graph=evidence_graph,
                **test.parameters,
            )

        elif test.capability == "vendor_alert":
            evidence = vendor_alert_capability.collect_evidence(
                evidence_graph=evidence_graph,
                **test.parameters,
            )

        elif test.capability == "deep_diagnostics":
            evidence = deep_diagnostics_capability.collect_evidence(
                evidence_graph=evidence_graph,
                **test.parameters,
            )

        else:
            raise ValueError(
                f"Unsupported capability: {test.capability}"
            )

        return EvidenceTestResult(
            test_id=test.test_id,
            hypothesis_id=test.hypothesis_id,
            capability=test.capability,
            status="completed",
            evidence=evidence,
        )

    except Exception as exc:
        return EvidenceTestResult(
            test_id=test.test_id,
            hypothesis_id=test.hypothesis_id,
            capability=test.capability,
            status="failed",
            error=str(exc),
        )

In [55]:
def execute_evidence_plan(
    evidence_plan: EvidencePlan,
    evidence_graph: EvidenceGraph,
    topology_capability: TopologyCapability,
    reachability_capability: ReachabilityCapability,
    network_state_capability: NetworkStateCapability,
    vendor_alert_capability: VendorAlertCapability,
    deep_diagnostics_capability: DeepDiagnosticsCapability,
) -> List[EvidenceTestResult]:

    ordered_tests = sorted(
        evidence_plan.tests,
        key=lambda test: test.priority,
    )

    results = []

    for test in ordered_tests:
        result = execute_evidence_test(
            test=test,
            evidence_graph=evidence_graph,
            topology_capability=topology_capability,
            reachability_capability=reachability_capability,
            network_state_capability=network_state_capability,
            vendor_alert_capability=vendor_alert_capability,
            deep_diagnostics_capability=deep_diagnostics_capability,
        )

        results.append(result)

    return results

##Vendor Alert Evidence Executor

In [56]:
def vendor_alert_evidence_executor(
    evidence_graph: EvidenceGraph,
    entity_ids: List[str],
) -> Dict[str, Any]:
    """
    Retrieve vendor-sourced alert/incident events for entities.

    Unlike network_state_evidence_executor, this does not filter by
    a fixed metric-name allowlist — vendor event_type values vary
    by source (PagerDuty's "vendor_incident", a future Datadog
    adapter's "monitor_alert", etc.), so all events touching the
    given entities are returned. Callers that care about a specific
    vendor or event type can filter the result further.
    """

    events = evidence_graph.get_events(
        entity_ids=entity_ids
    )

    return {
        "executor": "vendor_alert_evidence_executor",
        "entity_ids": list(entity_ids),
        "events": [event.to_dict() for event in events],
        "event_count": len(events),
    }

##DEEP DIAGNOSTICS EVIDENCE EXECUTOR

In [57]:
def deep_diagnostics_evidence_executor(
    evidence_graph: EvidenceGraph,
    entity_ids: List[str],
) -> Dict[str, Any]:
    """
    Fetch deep diagnostic telemetry (BGP session state, interface
    error counters, administrative status) for the given entities
    and add it to the evidence graph as it's retrieved.

    This is a deliberately separate, explicit query from
    network_state_evidence_executor — it represents evidence that
    is NOT part of routine first-response telemetry and must be
    specifically requested, mirroring a real investigator escalating
    from a dashboard check to pulling device-level diagnostics.

    Deterministic fixture, scoped to the NIKA simple_bgp scenario's
    injected router1<->router2 link failure. A later version would
    query NIKA's live diagnostic interfaces.
    """

    SOURCE_ID = "nika_simple_bgp_deep_diagnostics"
    OBSERVED_AT = "2026-08-07T00:00:03Z"

    fixture_observations_by_entity = {
        "nika:simple_bgp:router1": [
            ObservationRecord(
                observation_id="nika:simple_bgp:obs:router1-router2-bgp-state",
                source_id=SOURCE_ID,
                entity_id="nika:simple_bgp:router1",
                metric_name="bgp_session_state",
                metric_value="down",
                observed_at=OBSERVED_AT,
                dimensions={"peer_entity_id": "nika:simple_bgp:router2"},
            ),
            ObservationRecord(
                observation_id="nika:simple_bgp:obs:router1-router2-error-count",
                source_id=SOURCE_ID,
                entity_id="nika:simple_bgp:router1",
                metric_name="interface_error_count",
                metric_value=48213,
                observed_at=OBSERVED_AT,
                dimensions={"peer_entity_id": "nika:simple_bgp:router2"},
                unit="crc_errors",
            ),
            ObservationRecord(
                observation_id="nika:simple_bgp:obs:router1-router2-admin-status",
                source_id=SOURCE_ID,
                entity_id="nika:simple_bgp:router1",
                metric_name="interface_admin_status",
                metric_value="up",
                observed_at=OBSERVED_AT,
                dimensions={"peer_entity_id": "nika:simple_bgp:router2"},
            ),
        ],
        "nika:simple_bgp:router2": [
            ObservationRecord(
                observation_id="nika:simple_bgp:obs:router2-router1-bgp-state",
                source_id=SOURCE_ID,
                entity_id="nika:simple_bgp:router2",
                metric_name="bgp_session_state",
                metric_value="down",
                observed_at=OBSERVED_AT,
                dimensions={"peer_entity_id": "nika:simple_bgp:router1"},
            ),
            ObservationRecord(
                observation_id="nika:simple_bgp:obs:router2-router1-error-count",
                source_id=SOURCE_ID,
                entity_id="nika:simple_bgp:router2",
                metric_name="interface_error_count",
                metric_value=51002,
                observed_at=OBSERVED_AT,
                dimensions={"peer_entity_id": "nika:simple_bgp:router1"},
                unit="crc_errors",
            ),
            ObservationRecord(
                observation_id="nika:simple_bgp:obs:router2-router1-admin-status",
                source_id=SOURCE_ID,
                entity_id="nika:simple_bgp:router2",
                metric_name="interface_admin_status",
                metric_value="up",
                observed_at=OBSERVED_AT,
                dimensions={"peer_entity_id": "nika:simple_bgp:router1"},
            ),
        ],
    }

    new_observations = []

    for entity_id in entity_ids:
        for observation in fixture_observations_by_entity.get(entity_id, []):
            # Avoid duplicate inserts if this capability is called
            # more than once for the same entity across rounds.
            try:
                evidence_graph.add_observation(observation)
                new_observations.append(observation)
            except ValueError:
                # Already present from a prior round's call — that's fine,
                # still include it in this round's returned evidence.
                existing = evidence_graph.get_observations(
                    entity_ids=[entity_id],
                    metric_names=[observation.metric_name],
                )
                new_observations.extend(existing)

    return {
        "executor": "deep_diagnostics_evidence_executor",
        "entity_ids": list(entity_ids),
        "observations": [o.to_dict() for o in new_observations],
        "observation_count": len(new_observations),
    }


class DeepDiagnosticsCapability:
    """
    Answer the investigative question:

        What do device-level diagnostics (BGP session state,
        interface error counters, administrative status) show for
        these entities, beyond routine first-response telemetry?

    Unlike NetworkStateCapability, this data is not present in the
    graph until this capability is invoked — it represents an
    explicit escalation to deeper diagnostics, not passive
    telemetry that was always available.
    """

    def collect_evidence(
        self,
        evidence_graph: EvidenceGraph,
        entity_ids: List[str],
    ) -> Dict[str, Any]:

        entity_details = []

        for entity_id in entity_ids:
            entity_details.append(
                entity_lookup_executor(
                    evidence_graph=evidence_graph,
                    entity_id=entity_id,
                )
            )

        diagnostics = deep_diagnostics_evidence_executor(
            evidence_graph=evidence_graph,
            entity_ids=entity_ids,
        )

        evidence_status = (
            "available" if diagnostics["observation_count"] > 0
            else "insufficient_evidence"
        )

        return {
            "capability": "deep_diagnostics",
            "evidence_status": evidence_status,
            "entities": entity_details,
            "deep_diagnostics_evidence": diagnostics,
        }

#9. INVESTIGATION CAPABILITIES

Capabilities provide deterministic investigative services to AI agents.

Agents request information needs (e.g. topology, telemetry, events)
rather than invoking individual executors directly.

Capabilities may internally orchestrate one or more deterministic
executors and return a single normalized result.

##Topology Capability

In [58]:
from typing import Optional


class TopologyCapability:
    """
    Deterministic topology investigation capability.

    Provides a simplified interface over topology-related executors.
    """

    def collect_evidence(
        self,
        evidence_graph: EvidenceGraph,
        source_entity_id: str,
        target_entity_id: Optional[str] = None,
    ) -> Dict[str, Any]:

        result = {
            "capability": "topology",
        }

        # --------------------------------------------------
        # Source entity
        # --------------------------------------------------

        result["source_entity"] = entity_lookup_executor(
            evidence_graph=evidence_graph,
            entity_id=source_entity_id,
        )

        # --------------------------------------------------
        # Blast radius
        # --------------------------------------------------

        result["blast_radius"] = blast_radius_executor(
            evidence_graph=evidence_graph,
            failed_entity_id=source_entity_id,
        )

        # --------------------------------------------------
        # Optional path analysis
        # --------------------------------------------------

        if target_entity_id is not None:

            result["path_analysis"] = (
                path_analysis_executor(
                    evidence_graph=evidence_graph,
                    source_entity_id=source_entity_id,
                    target_entity_id=target_entity_id,
                )
            )

        return result

testing the topology capability

In [59]:
topology_capability = TopologyCapability()

topology_result = topology_capability.collect_evidence(
    evidence_graph=nika_graph,
    source_entity_id="nika:simple_bgp:router1",
    target_entity_id="nika:simple_bgp:pc2",
)

pprint.pprint(topology_result)

{'blast_radius': {'affected_count': 2,
                  'affected_entity_ids': ['nika:simple_bgp:pc2',
                                          'nika:simple_bgp:router2'],
                  'executor': 'blast_radius_executor',
                  'failed_entity': {'attributes': {'provider': 'nika',
                                                   'role': 'bgp_router',
                                                   'scenario': 'simple_bgp'},
                                    'entity_id': 'nika:simple_bgp:router1',
                                    'entity_type': 'device',
                                    'name': 'router1',
                                    'source_ids': ['nika_simple_bgp_topology']}},
 'capability': 'topology',
 'path_analysis': {'executor': 'path_analysis_executor',
                   'hop_count': 2,
                   'shortest_path': ['nika:simple_bgp:router1',
                                     'nika:simple_bgp:router2',
                            

##Reachability Capability

In [60]:
class ReachabilityCapability:
    """
    Answer the broad investigative question:

        Can entity A reach entity B, and what reachability
        evidence do we currently possess?

    The capability orchestrates deterministic executors only.
    """

    def collect_evidence(
        self,
        evidence_graph: EvidenceGraph,
        source_entity_id: str,
        target_entity_id: str,
    ) -> Dict[str, Any]:

        # Verify that both entities exist and capture context.
        source_entity = entity_lookup_executor(
            evidence_graph=evidence_graph,
            entity_id=source_entity_id,
        )

        target_entity = entity_lookup_executor(
            evidence_graph=evidence_graph,
            entity_id=target_entity_id,
        )

        # Structural path evidence.
        try:
            path_evidence = path_analysis_executor(
                evidence_graph=evidence_graph,
                source_entity_id=source_entity_id,
                target_entity_id=target_entity_id,
                include_all_paths=True,
            )

        except ValueError:
            path_evidence = {
                "path_exists": False,
                "shortest_path": None,
            }

        # Operational reachability evidence.
        reachability_evidence = (
            reachability_evidence_executor(
                evidence_graph=evidence_graph,
                source_entity_id=source_entity_id,
                target_entity_id=target_entity_id,
            )
        )

        if (
            reachability_evidence["observation_count"]
            == 0
        ):
            evidence_status = "insufficient_evidence"
        else:
            evidence_status = "available"

        return {
            "capability": "reachability",
            "evidence_status": evidence_status,
            "source_entity": source_entity,
            "target_entity": target_entity,
            "path_evidence": path_evidence,
            "reachability_evidence": (
                reachability_evidence
            ),
        }

Testing Reachability Capability

In [61]:
reachability_capability = ReachabilityCapability()

reachability_result = (
    reachability_capability.collect_evidence(
        evidence_graph=nika_graph,
        source_entity_id="nika:simple_bgp:pc1",
        target_entity_id="nika:simple_bgp:pc2",
    )
)

pprint.pprint(reachability_result)

{'capability': 'reachability',
 'evidence_status': 'insufficient_evidence',
 'path_evidence': {'all_paths': [['nika:simple_bgp:pc1',
                                  'nika:simple_bgp:router1',
                                  'nika:simple_bgp:router2',
                                  'nika:simple_bgp:pc2']],
                   'executor': 'path_analysis_executor',
                   'hop_count': 3,
                   'path_count': 1,
                   'shortest_path': ['nika:simple_bgp:pc1',
                                     'nika:simple_bgp:router1',
                                     'nika:simple_bgp:router2',
                                     'nika:simple_bgp:pc2'],
                   'source_entity_id': 'nika:simple_bgp:pc1',
                   'target_entity_id': 'nika:simple_bgp:pc2'},
 'reachability_evidence': {'executor': 'reachability_evidence_executor',
                           'observation_count': 0,
                           'observations': [],
             

In [62]:
print(nika_graph.summary())

for observation in nika_graph.get_observations():
    print(
        observation.metric_name,
        observation.entity_id,
        observation.metric_value,
        observation.dimensions,
    )

{'entities': 4, 'relationships': 3, 'observations': 5, 'events': 1, 'changes': 0}
interface_oper_status nika:simple_bgp:router1 down {'peer_entity_id': 'nika:simple_bgp:router2'}
interface_oper_status nika:simple_bgp:router2 down {'peer_entity_id': 'nika:simple_bgp:router1'}
ping_success nika:simple_bgp:router1 False {'target_entity_id': 'nika:simple_bgp:router2'}
ping_success nika:simple_bgp:pc1 True {'target_entity_id': 'nika:simple_bgp:router1'}
ping_success nika:simple_bgp:router2 True {'target_entity_id': 'nika:simple_bgp:pc2'}


In [63]:
reachability_result = reachability_capability.collect_evidence(
    evidence_graph=nika_graph,
    source_entity_id="nika:simple_bgp:router1",
    target_entity_id="nika:simple_bgp:router2",
)

pprint.pprint(reachability_result)

{'capability': 'reachability',
 'evidence_status': 'available',
 'path_evidence': {'all_paths': [['nika:simple_bgp:router1',
                                  'nika:simple_bgp:router2']],
                   'executor': 'path_analysis_executor',
                   'hop_count': 1,
                   'path_count': 1,
                   'shortest_path': ['nika:simple_bgp:router1',
                                     'nika:simple_bgp:router2'],
                   'source_entity_id': 'nika:simple_bgp:router1',
                   'target_entity_id': 'nika:simple_bgp:router2'},
 'reachability_evidence': {'executor': 'reachability_evidence_executor',
                           'observation_count': 1,
                           'observations': [{'dimensions': {'target_entity_id': 'nika:simple_bgp:router2'},
                                             'entity_id': 'nika:simple_bgp:router1',
                                             'metric_name': 'ping_success',
                             

##Network State Capability

In [64]:
class NetworkStateCapability:
    """
    Answer the broad investigative question:

        What operational network state is known for these entities?

    This may eventually cover interfaces, routing adjacencies,
    route presence, and device availability.
    """

    def collect_evidence(
        self,
        evidence_graph: EvidenceGraph,
        entity_ids: List[str],
    ) -> Dict[str, Any]:

        entity_details = []

        for entity_id in entity_ids:
            entity_details.append(
                entity_lookup_executor(
                    evidence_graph=evidence_graph,
                    entity_id=entity_id,
                )
            )

        network_state = (
            network_state_evidence_executor(
                evidence_graph=evidence_graph,
                entity_ids=entity_ids,
            )
        )

        if network_state["observation_count"] == 0:
            evidence_status = "insufficient_evidence"
        else:
            evidence_status = "available"

        return {
            "capability": "network_state",
            "evidence_status": evidence_status,
            "entities": entity_details,
            "network_state_evidence": network_state,
        }

Testing network state capability

In [65]:
network_state_capability = NetworkStateCapability()

network_state_result = (
    network_state_capability.collect_evidence(
        evidence_graph=nika_graph,
        entity_ids=[
            "nika:simple_bgp:router1",
            "nika:simple_bgp:router2",
        ],
    )
)

pprint.pprint(network_state_result)

{'capability': 'network_state',
 'entities': [{'entity': {'attributes': {'provider': 'nika',
                                         'role': 'bgp_router',
                                         'scenario': 'simple_bgp'},
                          'entity_id': 'nika:simple_bgp:router1',
                          'entity_type': 'device',
                          'name': 'router1',
                          'source_ids': ['nika_simple_bgp_topology']},
               'executor': 'entity_lookup_executor',
               'neighbors': [{'attributes': {'provider': 'nika',
                                             'role': 'host',
                                             'scenario': 'simple_bgp'},
                              'entity_id': 'nika:simple_bgp:pc1',
                              'entity_type': 'host',
                              'name': 'pc1',
                              'source_ids': ['nika_simple_bgp_topology']},
                             {'attributes': {'provide

##Vendor Alert Capability

In [66]:
class VendorAlertCapability:
    """
    Answer the broad investigative question:

        What have external monitoring/alerting vendors already
        reported about these entities?

    This surfaces vendor-sourced incidents (PagerDuty, and later
    Datadog or others) as evidence, but deliberately does not
    interpret or diagnose them — vendor alerts describe symptoms,
    not root cause. Interpretation is the Evidence Analyst's job,
    same as every other capability.
    """

    def collect_evidence(
        self,
        evidence_graph: EvidenceGraph,
        entity_ids: List[str],
    ) -> Dict[str, Any]:

        entity_details = []

        for entity_id in entity_ids:
            entity_details.append(
                entity_lookup_executor(
                    evidence_graph=evidence_graph,
                    entity_id=entity_id,
                )
            )

        vendor_alerts = (
            vendor_alert_evidence_executor(
                evidence_graph=evidence_graph,
                entity_ids=entity_ids,
            )
        )

        if vendor_alerts["event_count"] == 0:
            evidence_status = "insufficient_evidence"
        else:
            evidence_status = "available"

        return {
            "capability": "vendor_alert",
            "evidence_status": evidence_status,
            "entities": entity_details,
            "vendor_alert_evidence": vendor_alerts,
        }

testing the vendor alert capability

In [67]:
# testing vendor alert capability

vendor_alert_capability = VendorAlertCapability()

vendor_alert_result = (
    vendor_alert_capability.collect_evidence(
        evidence_graph=nika_graph,
        entity_ids=[
            "nika:simple_bgp:router1",
            "nika:simple_bgp:router2",
        ],
    )
)

pprint.pprint(vendor_alert_result)

{'capability': 'vendor_alert',
 'entities': [{'entity': {'attributes': {'provider': 'nika',
                                         'role': 'bgp_router',
                                         'scenario': 'simple_bgp'},
                          'entity_id': 'nika:simple_bgp:router1',
                          'entity_type': 'device',
                          'name': 'router1',
                          'source_ids': ['nika_simple_bgp_topology']},
               'executor': 'entity_lookup_executor',
               'neighbors': [{'attributes': {'provider': 'nika',
                                             'role': 'host',
                                             'scenario': 'simple_bgp'},
                              'entity_id': 'nika:simple_bgp:pc1',
                              'entity_type': 'host',
                              'name': 'pc1',
                              'source_ids': ['nika_simple_bgp_topology']},
                             {'attributes': {'provider

#10. AI AGENTS

##Incident Framing Agent

In [68]:
@observe(name="incident_framing_agent")
def incident_framing_agent(
    investigation_config: NetworkInvestigationConfig,
    evidence_graph: EvidenceGraph,
    topology_capability: TopologyCapability,
) -> IncidentFrame:
    """
    Frame the incident into a structured investigation problem.

    This agent must not diagnose root cause.
    """

    context = build_incident_framing_context(
        investigation_config=investigation_config,
        evidence_graph=evidence_graph,
        topology_capability=topology_capability,
    )

    system_prompt = """
You are the Incident Framing Agent for an autonomous network
investigation team.

Your job is to transform the reported network symptom into a precise,
bounded investigation problem.

You must distinguish:

- reported facts;
- assumptions;
- unknowns;
- affected entities;
- potentially relevant entities; and
- investigative capabilities that may be needed.

Do NOT diagnose root cause.
Do NOT recommend remediation.
Do NOT generate detailed hypotheses.

Return only a Python dictionary with exactly these keys:

{
    "investigation_question": str,
    "scope_summary": str,
    "known_facts": list[str],
    "assumptions": list[str],
    "unknowns": list[str],
    "affected_entity_ids": list[str],
    "potentially_relevant_entity_ids": list[str],
    "required_capabilities": list[str],
    "severity": str | None
}

Use only entity IDs that appear in the supplied context.
Only request capabilities listed under available_capabilities.
"""

    user_prompt = f"""
Investigation context:

{context}

Frame this network incident.
"""

    response = llm_call(
        agent_name="incident_framing_agent",
        messages=[{"role": "system",
              "content": system_prompt,},
            {"role": "user",
              "content": user_prompt,},
        ],temperature=0.1,
    )

    content = response["content"]


    if not content:
        raise ValueError(
            "Incident Framing Agent returned empty LLM content."
        )

    # --------------------------------------------------------
    # 5. Parse structured output
    # --------------------------------------------------------

    cleaned = clean_llm_dict_output(content)
    parsed = ast.literal_eval(cleaned)

    # --------------------------------------------------------
    # 6. Construct typed domain object
    # --------------------------------------------------------

    incident_frame = IncidentFrame(
        incident_id=(f"{investigation_config.investigation_id}-incident-frame"),
        investigation_question=parsed["investigation_question"],
        scope_summary=parsed[
            "scope_summary"
        ],
        known_facts=parsed.get(
            "known_facts",
            [],
        ),
        assumptions=parsed.get(
            "assumptions",
            [],
        ),
        unknowns=parsed.get(
            "unknowns",
            [],
        ),

        affected_entity_ids=parsed.get(
            "affected_entity_ids",
            [],
        ),

        potentially_relevant_entity_ids=parsed.get(
            "potentially_relevant_entity_ids",
            [],
        ),

        required_capabilities=parsed.get(
            "required_capabilities",
            [],
        ),

        severity=parsed.get(
            "severity"
        ),
    )

    # --------------------------------------------------------
    # 7. Validate
    # --------------------------------------------------------

    incident_frame.validate()

    # --------------------------------------------------------
    # 8. RETURN THE RESULT
    # --------------------------------------------------------

    return incident_frame



testing the incident framing agent

In [69]:
incident_frame = incident_framing_agent(
    investigation_config=nika_simple_bgp_config,
    evidence_graph=nika_graph,
    topology_capability=topology_capability,
)

pprint.pprint(incident_frame)

llm call completed
IncidentFrame(incident_id='nika-simple-bgp-link-down-001-incident-frame',
              investigation_question='Why is connectivity between pc1 and pc2 '
                                     'failing, and what is the state of the '
                                     'network path through router1 and '
                                     'router2?',
              scope_summary='Investigate connectivity failure between two '
                            'endpoint hosts (pc1 and pc2) that traverse a '
                            'BGP-routed core network. A vendor alert indicates '
                            '100% packet loss on the router1-router2 link. '
                            'Scope includes the two affected hosts and the '
                            'intermediate routing infrastructure.',
              known_facts=['Users report connectivity issues between pc1 and '
                           'pc2',
                           'Incident started at 2026-08-07T

##Hypothesis Generator Agent

In [70]:
@observe(name="hypothesis_generator_agent")
def hypothesis_generator_agent(
    incident_frame: IncidentFrame,
    evidence_graph: EvidenceGraph,
    topology_capability: TopologyCapability,
) -> HypothesisSet:
    """
    Generate competing, falsifiable explanations for the incident.

    This agent proposes hypotheses only.
    It does not decide which hypothesis is correct.
    """

    agent_name = "hypothesis_generator_agent"

    context = build_hypothesis_generation_context(
        incident_frame=incident_frame,
        evidence_graph=evidence_graph,
        topology_capability=topology_capability,
    )

    system_prompt = """
You are the Hypothesis Generator Agent for an autonomous
network investigation team.

Your responsibility is to generate multiple competing,
falsifiable explanations for a framed network incident.

Important rules:

1. Generate between 3 and 5 competing hypotheses.
2. Do NOT select a root cause.
3. Do NOT recommend remediation.
4. Each hypothesis must explain a plausible causal mechanism.
5. Each hypothesis must specify observable evidence that would
   support it.
6. Each hypothesis must specify at least one observation that
   would falsify it.
7. Use only entity IDs present in the supplied context.
8. Request only capabilities listed under available_capabilities.
9. Do not treat benchmark ground truth as investigation evidence.
10. Avoid duplicate hypotheses that describe the same failure
    using different wording.

Return ONLY a Python dictionary with exactly this shape:

{
    "hypotheses": [
        {
            "title": str,
            "proposed_cause": str,
            "causal_mechanism": str,
            "suspected_entity_ids": list[str],
            "expected_observations": list[str],
            "falsifying_observations": list[str],
            "required_capabilities": list[str],
            "prior_confidence": float
        }
    ]
}

prior_confidence must be between 0.0 and 1.0.

The confidence values represent initial plausibility only.
They do not need to sum to 1.0.
"""

    user_prompt = f"""
Investigation context:

{context}

Generate competing hypotheses for this incident.
"""

    response = llm_call(
        agent_name=agent_name,
        messages=[
            {"role": "system",
                "content": system_prompt,},
            {"role": "user",
                "content": user_prompt,},],
        temperature=0.2,)

    print("Hypothesis generation LLM call completed")

    content = response["content"]

    if not content:
        raise ValueError(
            "Hypothesis Generator Agent returned empty LLM content."
        )


    parsed = parse_or_repair_agent_response(
        raw_output=content,
        expected_schema=HYPOTHESIS_SET_SCHEMA,
        agent_name=agent_name,
    )

    raw_hypotheses = parsed.get(
        "hypotheses",
        []
    )

    hypotheses = []

    for index, item in enumerate(
        raw_hypotheses,
        start=1,
    ):
        hypothesis = HypothesisRecord(
            hypothesis_id=(
                f"{incident_frame.incident_id}-h{index}"
            ),

            incident_id=incident_frame.incident_id,

            title=item["title"],

            proposed_cause=item[
                "proposed_cause"
            ],

            causal_mechanism=item[
                "causal_mechanism"
            ],

            suspected_entity_ids=item.get(
                "suspected_entity_ids",
                [],
            ),
            expected_observations=item.get(
                "expected_observations",
                [],
            ),
            falsifying_observations=item.get(
                "falsifying_observations",
                [],
            ),
            required_capabilities=item.get(
                "required_capabilities",
                [],
            ),
            prior_confidence=float(
                item.get(
                    "prior_confidence",
                    0.0,
                )
            ),
        )

        hypothesis.validate()
        hypotheses.append(hypothesis)

    hypothesis_set = HypothesisSet(
        incident_id=incident_frame.incident_id,
        hypotheses=hypotheses,
    )

    hypothesis_set.validate()

    return hypothesis_set

testing the hypothesis generator agent

In [71]:
hypothesis_set = hypothesis_generator_agent(
    incident_frame=incident_frame,
    evidence_graph=nika_graph,
    topology_capability=topology_capability,
)

pprint.pprint(hypothesis_set)

llm call completed
Hypothesis generation LLM call completed
HypothesisSet(incident_id='nika-simple-bgp-link-down-001-incident-frame',
              hypotheses=[HypothesisRecord(hypothesis_id='nika-simple-bgp-link-down-001-incident-frame-h1',
                                           incident_id='nika-simple-bgp-link-down-001-incident-frame',
                                           title='Physical Layer 1 Link '
                                                 'Failure on router1-router2 '
                                                 'Interface',
                                           proposed_cause='The router1-router2 '
                                                          'link has '
                                                          'experienced a '
                                                          'physical layer '
                                                          'failure (e.g., '
                                                          'fib

##Evidence Planning Agent

In [108]:
@observe(name="evidence_planning_agent")
def evidence_planning_agent(
    incident_frame: IncidentFrame,
    hypothesis_set: HypothesisSet,
    round_number: int = 1,
    prior_evidence_analysis: Optional[EvidenceAnalysis] = None,
    prior_challenge_report: Optional[ChallengeReport] = None,
) -> EvidencePlan:
    """
    Convert competing hypotheses into explicit capability requests.

    This agent plans evidence collection only.
    It does not execute tests or judge hypotheses.

    round_number identifies which reasoning-loop pass this is
    (1 for the first round). It is used only to keep test_id unique
    across rounds within one investigation.

    When prior_evidence_analysis / prior_challenge_report are
    supplied, this is a subsequent reasoning-loop round: planning
    should prioritize the gaps those identified over re-deriving
    tests from the hypotheses alone.
    """

    agent_name = "evidence_planning_agent"

    context = build_evidence_planning_context(
        incident_frame=incident_frame,
        hypothesis_set=hypothesis_set,
        prior_evidence_analysis=prior_evidence_analysis,
        prior_challenge_report=prior_challenge_report,
    )

    is_subsequent_round = (
        prior_evidence_analysis is not None
        or prior_challenge_report is not None
    )

    system_prompt = """
You are the Evidence Planning Agent for an autonomous network
investigation team.

Your responsibility is to convert each hypothesis into a minimal set
of deterministic evidence tests.

Important rules:

1. Do NOT decide whether a hypothesis is correct.
2. Do NOT collect evidence yourself.
3. Do NOT invoke executors directly.
4. Request only capabilities listed under available_capabilities.
5. Every test must have a clear objective.
6. Every test must identify which hypothesis it evaluates.
7. Reuse a test when one evidence request can evaluate multiple ideas,
   but each returned test must reference one hypothesis_id.
8. Prefer the smallest set of high-information tests. Maximum 2
   tests per hypothesis per round — if you find yourself writing a
   3rd test for the same hypothesis, combine it into one of the
   first 2 by broadening that test's objective instead.
9. Use only entity IDs present in the supplied context.
10. Parameters must contain only values needed by the capability.
11. Do not invent capability names.
12. Do not use benchmark ground truth.
13. A single test's objective may ask a capability to report on
    MULTIPLE related diagnostic questions at once (e.g. "check
    error counters, administrative status, and signal quality on
    this interface" is ONE test, not three) — capabilities return
    whatever evidence they have for the requested entities, so
    splitting related questions about the same entity into separate
    tests wastes budget without adding information.

If prior_round_missing_evidence or prior_round_challenger_gaps are
present in the context, this is NOT the first planning round for
this investigation. Evidence has already been collected and
analyzed, and it was not sufficient. In this case:

14. Prioritize tests that would resolve the specific gaps listed in
    prior_round_challenger_gaps first — these come from a
    dedicated adversarial review of the leading hypothesis, and
    are the highest-value gaps to close.
15. Then address gaps in prior_round_missing_evidence not already
    covered by a test targeting the challenger gaps.
16. Do NOT repeat a test that would collect the same evidence as a
    prior round already attempted and found insufficient — the gap
    exists because that evidence is genuinely unavailable or was
    already checked, not because no one asked. Plan a DIFFERENT
    test/capability/parameters that could close the gap another way,
    or state within the objective why no further test can close it.
17. It is acceptable for this round's plan to contain fewer tests
    than the hypothesis count, if only a few real gaps remain.
18. The max-2-tests-per-hypothesis rule (rule 8) still applies here.
    If more than 2 gaps exist for one hypothesis, prioritize the
    single highest-value gap and combine the rest into that same
    test's objective — do NOT write one test per listed gap.

Return ONLY a Python dictionary with exactly this shape:

{
    "tests": [
        {
            "hypothesis_id": str,
            "objective": str,
            "capability": str,
            "parameters": dict,
            "expected_supporting_observations": list[str],
            "expected_falsifying_observations": list[str],
            "priority": int
        }
    ]
}

For topology and reachability tests, parameters may include:
{
    "source_entity_id": str,
    "target_entity_id": str
}

For network_state tests, parameters may include:
{
    "entity_ids": list[str]
}

Priority 1 means highest priority.
"""

    user_prompt = f"""
Investigation context:

{context}

{"This is a subsequent reasoning round. Prior evidence was insufficient to reach a confident, unchallenged conclusion. Focus this round's plan on closing the specific gaps identified above." if is_subsequent_round else ""}

Create the evidence collection plan.
"""

    response = llm_call(
        agent_name=agent_name,
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ],
        temperature=0.1,
    )

    print("Evidence planning LLM call completed")

    content = response["content"]

    if not content:
        raise ValueError(
            "Evidence Planning Agent returned empty LLM content."
        )

    parsed = parse_or_repair_agent_response(
        raw_output=content,
        expected_schema=EVIDENCE_PLAN_SCHEMA,
        agent_name=agent_name,
    )

    raw_tests = parsed.get("tests", [])

    # ------------------------------------------------------
    # Step 1: filter out items missing required fields
    # ------------------------------------------------------

    REQUIRED_TEST_FIELDS = ("hypothesis_id", "objective", "capability")

    valid_raw_tests = []

    for i, item in enumerate(raw_tests):
        missing_fields = [
            field for field in REQUIRED_TEST_FIELDS
            if field not in item or not item[field]
        ]

        if missing_fields:
            print(
                f"⚠ evidence_planning_agent test #{i + 1} is missing "
                f"required field(s) {missing_fields} — skipping it: "
                f"{item}"
            )
            continue

        valid_raw_tests.append(item)

    raw_tests = valid_raw_tests

    # ------------------------------------------------------
    # Step 2: normalize hypothesis_id and group by hypothesis
    # ------------------------------------------------------

    raw_tests_by_hypothesis: Dict[str, List[Dict[str, Any]]] = {}

    for item in raw_tests:
        try:
            normalized_id = normalize_hypothesis_id(
                returned_id=item["hypothesis_id"],
                hypothesis_set=hypothesis_set,
            )
        except ValueError:
            print(
                f"⚠ evidence_planning_agent test has unresolvable "
                f"hypothesis_id '{item['hypothesis_id']}' — skipping it"
            )
            continue

        item["hypothesis_id"] = normalized_id
        raw_tests_by_hypothesis.setdefault(normalized_id, []).append(item)

    # ------------------------------------------------------
    # Step 3: deterministic cap — max N tests per hypothesis,
    # kept by priority. Backstop against the model ignoring
    # rule 8/18, which it has done multiple times.
    # ------------------------------------------------------

    MAX_TESTS_PER_HYPOTHESIS = 2

    def _priority(item: Dict[str, Any]) -> int:
        try:
            return int(item.get("priority", 1))
        except (TypeError, ValueError):
            return 1

    capped_raw_tests = []

    for hyp_id, items in raw_tests_by_hypothesis.items():
        sorted_items = sorted(items, key=_priority)
        capped_raw_tests.extend(sorted_items[:MAX_TESTS_PER_HYPOTHESIS])

        if len(sorted_items) > MAX_TESTS_PER_HYPOTHESIS:
            print(
                f"⚠ evidence_planning_agent requested "
                f"{len(sorted_items)} tests for hypothesis "
                f"'{hyp_id}' — capping to "
                f"{MAX_TESTS_PER_HYPOTHESIS} by priority"
            )

    raw_tests = capped_raw_tests

    if not raw_tests:
        raise ValueError(
            "Evidence Planning Agent: no valid tests remained after "
            "filtering malformed items."
        )

    # ------------------------------------------------------
    # Step 4: build EvidenceTest objects
    # ------------------------------------------------------

    tests = []

    for index, item in enumerate(
        raw_tests,
        start=1,
    ):
        test = EvidenceTest(
            test_id=(
                f"{incident_frame.incident_id}-r{round_number}-test-{index}"
            ),

            hypothesis_id=item["hypothesis_id"],

            objective=item["objective"],

            capability=item["capability"],

            parameters=item.get(
                "parameters",
                {},
            ),

            expected_supporting_observations=item.get(
                "expected_supporting_observations",
                [],
            ),

            expected_falsifying_observations=item.get(
                "expected_falsifying_observations",
                [],
            ),

            priority=int(
                item.get("priority", 1)
            ),
        )

        test.validate()
        tests.append(test)

    evidence_plan = EvidencePlan(
        incident_id=incident_frame.incident_id,
        tests=tests,
    )

    evidence_plan.validate()

    return evidence_plan

##Evidence Planning Agent

In [73]:
evidence_plan = evidence_planning_agent(
    incident_frame=incident_frame,
    hypothesis_set=hypothesis_set,
)

for test in evidence_plan.tests:
    print("=" * 70)
    print("Test:", test.test_id)
    print("Hypothesis:", test.hypothesis_id)
    print("Objective:", test.objective)
    print("Capability:", test.capability)
    print("Parameters:", test.parameters)
    print("Priority:", test.priority)

llm call completed
Evidence planning LLM call completed
Test: nika-simple-bgp-link-down-001-incident-frame-r1-test-1
Hypothesis: nika-simple-bgp-link-down-001-incident-frame-h1
Objective: Determine if the router1-router2 link has experienced a physical Layer 1 failure by checking interface operational status and carrier signal presence on both ends of the link.
Capability: network_state
Parameters: {'entity_ids': ['nika:simple_bgp:router1', 'nika:simple_bgp:router2']}
Priority: 1
Test: nika-simple-bgp-link-down-001-incident-frame-r1-test-2
Hypothesis: nika-simple-bgp-link-down-001-incident-frame-h2
Objective: Determine if the BGP session between router1 and router2 has failed or become unstable by checking BGP session state, neighbor adjacency status, and route presence in routing tables.
Capability: deep_diagnostics
Parameters: {'entity_ids': ['nika:simple_bgp:router1', 'nika:simple_bgp:router2']}
Priority: 2
Test: nika-simple-bgp-link-down-001-incident-frame-r1-test-3
Hypothesis: nik

In [74]:
EVIDENCE_ANALYST_BANNED_PHRASES = [
    "root cause",
    "confirmed",
    "definitively",
]


def check_evidence_analysis_leakage(
    evidence_analysis: EvidenceAnalysis,
) -> List[str]:
    """
    Deterministic guardrail: flag any assessment text that leaks
    root-cause language the Evidence Analyst is contractually
    forbidden from using.

    Returns a list of violation descriptions. An empty list means
    no violations were found.
    """

    violations = []

    text_fields = (
        "rationale",
        "supporting_evidence",
        "contradicting_evidence",
        "missing_evidence",
    )

    for assessment in evidence_analysis.assessments:
        for field_name in text_fields:
            value = getattr(assessment, field_name)

            if isinstance(value, list):
                haystack = " ".join(value).lower()
            else:
                haystack = str(value).lower()

            for phrase in EVIDENCE_ANALYST_BANNED_PHRASES:
                if phrase in haystack:
                    violations.append(
                        f"{assessment.hypothesis_id}.{field_name} "
                        f"contains banned phrase '{phrase}'"
                    )

    return violations

In [75]:
def normalize_hypothesis_id(
    returned_id: str,
    hypothesis_set: HypothesisSet,
) -> str:
    """
    Normalize shorthand hypothesis IDs such as "H1", "h1", "1",
    "H1: BGP Configuration Error", or a bare hypothesis title to
    the canonical hypothesis_id used by the investigation.
    """

    canonical_ids = {
        hypothesis.hypothesis_id
        for hypothesis in hypothesis_set.hypotheses
    }

    if returned_id in canonical_ids:
        return returned_id

    normalized = returned_id.strip().lower()

    # Try a digit-based match first (h1, H1: title, bare "1", etc.)
    match = re.search(r"\d+", normalized)

    if match:
        index = int(match.group(0))
        candidate = f"{hypothesis_set.incident_id}-h{index}"

        if candidate in canonical_ids:
            return candidate

    # Fall back to matching against the hypothesis's own title —
    # covers the model returning a bare title with no numeric
    # reference at all.
    for hypothesis in hypothesis_set.hypotheses:
        if hypothesis.title.strip().lower() == normalized:
            return hypothesis.hypothesis_id

    raise ValueError(
        "Evidence Analyst returned unresolvable hypothesis_id: "
        f"{returned_id}"
    )

In [76]:
@observe(name="evidence_analyst_agent")
def evidence_analyst_agent(
    incident_frame: IncidentFrame,
    hypothesis_set: HypothesisSet,
    evidence_plan: EvidencePlan,
    evidence_results: List[EvidenceTestResult],
) -> EvidenceAnalysis:
    """
    Evaluate each hypothesis against its collected evidence.

    The Evidence Analyst:
      - identifies supporting evidence;
      - identifies contradicting evidence;,
      - identifies missing evidence; and
      - assesses hypothesis status.

    It does NOT determine root cause or recommend remediation.
    """

    agent_name = "evidence_analyst_agent"

    # --------------------------------------------------------
    # 1. Build bounded hypothesis-specific evidence context
    # --------------------------------------------------------

    context = build_evidence_analysis_context(
        hypothesis_set=hypothesis_set,
        evidence_plan=evidence_plan,
        evidence_results=evidence_results,
    )

    # --------------------------------------------------------
    # 2. Define agent instructions
    # --------------------------------------------------------

    system_prompt = """
You are the Evidence Analyst Agent for an autonomous network
investigation team.

You evaluate existing hypotheses against collected evidence.

You are NOT the Root Cause Agent.

For each hypothesis independently:

1. Review ONLY the tests listed under that hypothesis.
2. Identify observations that support the hypothesis.
3. Identify observations that contradict the hypothesis.
4. Identify important evidence that remains unavailable.
5. Assign exactly one status:
   - supported
   - weakened
   - rejected
   - proposed

Use "proposed" when the evidence is insufficient to evaluate the
hypothesis materially.

Critical rules:

- Never use evidence belonging to another hypothesis.
- Never invent observations.
- A successful test execution is not evidence by itself.
- "insufficient_evidence" means missing evidence.
- A structural topology path does not prove operational reachability.
- Successful reachability may falsify a hypothesis asserting that
  same segment is unreachable.
- Do NOT determine root cause.
- Do NOT recommend remediation.
- Do NOT use words such as "confirmed" or "root cause".
- Do NOT generate new hypotheses.
- Do NOT reference other hypotheses by ID (e.g. "H2") or compare
  one hypothesis's plausibility against another's — assess each
  hypothesis strictly against its own evidence in isolation.
- Use only the supplied evidence.
- Confidence must be between 0.0 and 1.0.

Return ONLY a Python dictionary.
No Markdown.
No headings.
No prose before or after the dictionary.
No code fences.

The dictionary must have exactly this shape:

{
    "assessments": [
        {
            "hypothesis_id": str,
            "status": str,
            "confidence": float,
            "supporting_evidence": list[str],
            "contradicting_evidence": list[str],
            "missing_evidence": list[str],
            "rationale": str
        }
    ]
}
"""

    # --------------------------------------------------------
    # 3. Invoke the LLM
    # --------------------------------------------------------

    response = llm_call(
        agent_name=agent_name,
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": f"""
                Incident frame:

                {incident_frame.to_dict()}

                Evidence analysis context:

                {context}

                Evaluate every hypothesis.
                """,
            },
        ],
        temperature=0.1,
    )

    print("Evidence analysis LLM call completed")

    # --------------------------------------------------------
    # 4. Parse structured response
    # --------------------------------------------------------

    parsed = parse_or_repair_agent_response(
        raw_output=response["content"],
      expected_schema=EVIDENCE_ANALYSIS_SCHEMA,
      agent_name=agent_name,)


    valid_hypothesis_ids = {
    hypothesis.hypothesis_id
    for hypothesis in hypothesis_set.hypotheses}

    valid_statuses = {
        "supported",
        "weakened",
        "rejected",
        "proposed",}

    for item in parsed.get("assessments", []):

      item["hypothesis_id"] = normalize_hypothesis_id(
          returned_id=item["hypothesis_id"],
          hypothesis_set=hypothesis_set,
      )

      if item["status"] not in valid_statuses:
        raise ValueError(
            "Evidence Analyst returned invalid status: "
            f"{item['status']}")

    # --------------------------------------------------------
    # 5. Construct typed assessments
    # --------------------------------------------------------

    assessments = []

    for item in parsed.get("assessments", []):

        assessment = HypothesisAssessment(
            hypothesis_id=item["hypothesis_id"],
            status=HypothesisStatus(
                item["status"]
            ),
            confidence=float(
                item["confidence"]
            ),
            supporting_evidence=item.get(
                "supporting_evidence",
                [],
            ),
            contradicting_evidence=item.get(
                "contradicting_evidence",
                [],
            ),
            missing_evidence=item.get(
                "missing_evidence",
                [],
            ),
            rationale=item.get(
                "rationale",
                "",
            ),
        )

        assessment.validate()

        assessments.append(
            assessment
        )

    # --------------------------------------------------------
    # 6. Construct complete evidence analysis
    # --------------------------------------------------------

    analysis = EvidenceAnalysis(
        incident_id=incident_frame.incident_id,
        assessments=assessments,
    )

    analysis.validate()

    return analysis

##HYPOTHESIS CHALLENGER AGENT

In [77]:
# ============================================================
# HYPOTHESIS CHALLENGER — DOMAIN OBJECTS
# ============================================================

@dataclass
class HypothesisChallenge:
    """
    Adversarial review of one hypothesis assessment.

    The Challenger does not re-score confidence independently — it
    either upholds the Evidence Analyst's assessment or disputes it
    with a specific rebuttal grounded in the same evidence.
    """

    hypothesis_id: str
    original_status: str
    original_confidence: float

    challenge_outcome: str  # "upheld" | "disputed" | "unresolved"
    rebuttal: str

    alternative_explanation: Optional[str] = None
    additional_evidence_needed: List[str] = field(default_factory=list)

    def validate(self) -> None:
        if not self.hypothesis_id.strip():
            raise ValueError("hypothesis_id cannot be empty.")

        valid_outcomes = {"upheld", "disputed", "unresolved"}

        if self.challenge_outcome not in valid_outcomes:
            raise ValueError(
                "challenge_outcome must be one of "
                f"{sorted(valid_outcomes)}, got "
                f"'{self.challenge_outcome}'."
            )

        if not self.rebuttal.strip():
            raise ValueError("rebuttal cannot be empty.")

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass
class ChallengeReport:
    """
    Full adversarial review of an investigation's evidence analysis.
    """

    incident_id: str
    challenges: List[HypothesisChallenge]

    systemic_concerns: List[str] = field(default_factory=list)

    def validate(self) -> None:
        if not self.incident_id.strip():
            raise ValueError("incident_id cannot be empty.")

        if not self.challenges:
            raise ValueError(
                "ChallengeReport must contain at least one challenge."
            )

        for challenge in self.challenges:
            challenge.validate()

    def to_dict(self) -> Dict[str, Any]:
        return {
            "incident_id": self.incident_id,
            "challenges": [c.to_dict() for c in self.challenges],
            "systemic_concerns": list(self.systemic_concerns),
        }

In [78]:
# ============================================================
# HYPOTHESIS CHALLENGER — CONTEXT BUILDER
# ============================================================

def rank_hypothesis_assessments(
    evidence_analysis: EvidenceAnalysis,
) -> List[HypothesisAssessment]:
    """
    Deterministically rank assessments — this is code, not an LLM
    call, so ranking is reproducible and the LLM never has to
    invent an ordering from scratch.
    """

    status_priority = {
        "supported": 0,
        "proposed": 1,
        "weakened": 2,
        "rejected": 3,
    }

    return sorted(
        evidence_analysis.assessments,
        key=lambda a: (
            status_priority.get(a.status.value, 99),
            -a.confidence,
        ),
    )


def build_challenger_context(
    incident_frame: IncidentFrame,
    hypothesis_set: HypothesisSet,
    evidence_analysis: EvidenceAnalysis,
) -> Dict[str, Any]:
    """
    Build bounded context for the Hypothesis Challenger.

    Only the top-ranked and runner-up assessments are passed in
    full detail — the Challenger's job is to stress-test the
    leading explanation(s), not re-review everything from scratch.
    """

    ranked = rank_hypothesis_assessments(evidence_analysis)

    hypotheses_by_id = {
        h.hypothesis_id: h
        for h in hypothesis_set.hypotheses
    }

    def assessment_package(assessment: HypothesisAssessment) -> Dict[str, Any]:
        hypothesis = hypotheses_by_id.get(assessment.hypothesis_id)

        return {
            "assessment": assessment.to_dict(),
            "hypothesis": (
                hypothesis.to_dict() if hypothesis else None
            ),
        }

    return {
        "incident_frame": incident_frame.to_dict(),

        "leading_assessment": (
            assessment_package(ranked[0]) if ranked else None
        ),

        "runner_up_assessment": (
            assessment_package(ranked[1]) if len(ranked) > 1 else None
        ),

        "all_assessment_statuses": [
            {
                "hypothesis_id": a.hypothesis_id,
                "status": a.status.value,
                "confidence": a.confidence,
            }
            for a in ranked
        ],
    }

In [79]:
CHALLENGE_REPORT_SCHEMA = """
{
    "challenges": [
        {
            "hypothesis_id": str,
            "challenge_outcome": str,
            "rebuttal": str,
            "alternative_explanation": str | None,
            "additional_evidence_needed": list[str]
        }
    ],
    "systemic_concerns": list[str]
}
"""

In [80]:
@observe(name="hypothesis_challenger_agent")
def hypothesis_challenger_agent(
    incident_frame: IncidentFrame,
    hypothesis_set: HypothesisSet,
    evidence_analysis: EvidenceAnalysis,
) -> ChallengeReport:
    """
    Adversarially review the leading hypothesis assessment(s).

    This agent argues against the current leading explanation using
    only the supplied evidence. It does not select a root cause or
    recommend remediation.
    """

    agent_name = "hypothesis_challenger_agent"

    context = build_challenger_context(
        incident_frame=incident_frame,
        hypothesis_set=hypothesis_set,
        evidence_analysis=evidence_analysis,
    )

    system_prompt = """
You are the Hypothesis Challenger Agent for an autonomous network
investigation team.

Your job is to argue AGAINST the leading hypothesis assessment,
using only the evidence already supplied. You are a devil's
advocate, not a second Evidence Analyst.

For the leading_assessment and runner_up_assessment supplied:

1. Identify the strongest counter-argument the evidence allows,
   even if you believe the original assessment is likely correct.
2. State whether the original assessment is "upheld" (you found no
   genuine counter-argument), "disputed" (you found a real
   weakness), or "unresolved" (evidence is too thin to argue either
   way).
3. If you dispute an assessment, name a specific alternative
   explanation the evidence would also be consistent with.
4. List any additional evidence that would resolve the dispute.

Also assess systemic_concerns across the whole investigation, such
as: was only one hypothesis meaningfully tested; did evidence
planning favor the leading hypothesis; are there hypotheses with
"proposed" status that were never actually challenged.

Critical rules:

- Do NOT determine root cause.
- Do NOT recommend remediation.
- Do NOT invent evidence not present in the supplied context.
- Every hypothesis passed to you (leading and runner-up, if
  present) must receive exactly one challenge entry.
- A rebuttal must reference specific evidence, not general doubt.

Return ONLY a Python dictionary with exactly this shape:

{
    "challenges": [
        {
            "hypothesis_id": str,
            "challenge_outcome": str,
            "rebuttal": str,
            "alternative_explanation": str | None,
            "additional_evidence_needed": list[str]
        }
    ],
    "systemic_concerns": list[str]
}

challenge_outcome must be exactly one of: "upheld", "disputed",
"unresolved".
"""

    user_prompt = f"""
Investigation context:

{context}

Challenge the leading hypothesis assessment(s).
"""

    response = llm_call(
        agent_name=agent_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.3,
    )

    print("Hypothesis challenge LLM call completed")

    content = response["content"]

    if not content:
        raise ValueError(
            "Hypothesis Challenger Agent returned empty LLM content."
        )

    parsed = parse_or_repair_agent_response(
        raw_output=content,
        expected_schema=CHALLENGE_REPORT_SCHEMA,
        agent_name=agent_name,
    )

    challenges = []

    for item in parsed.get("challenges", []):
        challenge = HypothesisChallenge(
            hypothesis_id=item["hypothesis_id"],
            original_status="",  # filled in below
            original_confidence=0.0,
            challenge_outcome=item["challenge_outcome"],
            rebuttal=item["rebuttal"],
            alternative_explanation=item.get("alternative_explanation"),
            additional_evidence_needed=item.get(
                "additional_evidence_needed", []
            ),
        )
        challenges.append(challenge)

    # Backfill original status/confidence from the source assessments
    # rather than trusting the LLM to echo them accurately.
    assessments_by_id = {
        a.hypothesis_id: a for a in evidence_analysis.assessments
    }

    for challenge in challenges:
        source = assessments_by_id.get(challenge.hypothesis_id)

        if source is not None:
            challenge.original_status = source.status.value
            challenge.original_confidence = source.confidence

        challenge.validate()

    report = ChallengeReport(
        incident_id=incident_frame.incident_id,
        challenges=challenges,
        systemic_concerns=parsed.get("systemic_concerns", []),
    )

    report.validate()

    return report

##ROOT CAUSE REMEDIATION AGENT

Helpers

In [81]:
def directional_confidence(assessment: HypothesisAssessment) -> float:
    """
    Return confidence as "likelihood this hypothesis is the true
    cause", normalized regardless of how the Evidence Analyst
    happened to interpret confidence direction this run.

    supported / weakened -> confidence already points toward
        "likely the cause", used as-is.
    rejected -> confidence is expected to point toward "likely NOT
        the cause" per the prompt, but the Evidence Analyst has
        been observed inverting this. Since a rejected hypothesis
        should never be treated as a plausible cause regardless of
        its reported number, this is deterministically floored to
        0.0 rather than trusted.
    proposed -> insufficient evidence to say either way; treated as
        unknown, not zero.
    """

    if assessment.status == HypothesisStatus.REJECTED:
        return 0.0

    if assessment.status == HypothesisStatus.PROPOSED:
        return assessment.confidence  # informational only, not a claim of likelihood

    return assessment.confidence  # supported / weakened

In [82]:
def select_leading_hypothesis(
    evidence_analysis: EvidenceAnalysis,
    minimum_confidence: float,
) -> Optional[HypothesisAssessment]:
    """
    Deterministically select the strongest candidate root cause.

    Only "supported" hypotheses are eligible, and only if they clear
    the configured minimum_diagnosis_confidence threshold. This is
    pre-filtering done in code, not by the LLM, so the LLM's job
    narrows to explaining and recommending — not ranking.
    """

    supported = [
        a for a in evidence_analysis.assessments
        if a.status == HypothesisStatus.SUPPORTED
        and directional_confidence(a) >= minimum_confidence
    ]

    if not supported:
        return None

    return max(supported, key=directional_confidence)

Root Cause & Remediation Context Builder

In [83]:
# ============================================================
# ROOT CAUSE & REMEDIATION — CONTEXT BUILDER
# ============================================================

def build_root_cause_context(
    incident_frame: IncidentFrame,
    hypothesis_set: HypothesisSet,
    leading_assessment: HypothesisAssessment,
    challenge_report: Optional[ChallengeReport],
) -> Dict[str, Any]:
    """
    Build bounded context for Root Cause & Remediation.

    Only the leading (already-selected) hypothesis and its matching
    Challenger verdict are included — this agent explains and
    recommends, it does not re-select among hypotheses.
    """

    hypotheses_by_id = {h.hypothesis_id: h for h in hypothesis_set.hypotheses}
    leading_hypothesis = hypotheses_by_id.get(leading_assessment.hypothesis_id)

    matching_challenge = None
    if challenge_report is not None:
        for c in challenge_report.challenges:
            if c.hypothesis_id == leading_assessment.hypothesis_id:
                matching_challenge = c.to_dict()
                break

    return {
        "incident_frame": incident_frame.to_dict(),
        "leading_hypothesis": (
            leading_hypothesis.to_dict() if leading_hypothesis else None
        ),
        "leading_assessment": leading_assessment.to_dict(),
        "challenger_verdict": matching_challenge,  # None if not challenged, or no matching entry
    }

In [84]:
@observe(name="root_cause_remediation_agent")
def root_cause_remediation_agent(
    incident_frame: IncidentFrame,
    hypothesis_set: HypothesisSet,
    evidence_analysis: EvidenceAnalysis,
    challenge_report: Optional[ChallengeReport],
    investigation_policy: InvestigationPolicy,
) -> RootCauseRecommendation:
    """
    Commit to a root cause and recommend remediation, or explicitly
    report insufficient evidence.

    Hypothesis selection is deterministic (select_leading_hypothesis)
    and happens in code before this agent is called. The LLM's job
    is narrower: explain the chosen cause in plain language, address
    the Challenger's dispute if one exists, and propose concrete
    remediation steps.
    """

    agent_name = "root_cause_remediation_agent"

    leading_assessment = select_leading_hypothesis(
        evidence_analysis=evidence_analysis,
        minimum_confidence=investigation_policy.minimum_diagnosis_confidence,
    )

    if leading_assessment is None:
        recommendation = RootCauseRecommendation(
            incident_id=incident_frame.incident_id,
            status="insufficient_evidence",
        )
        recommendation.validate()
        return recommendation

    context = build_root_cause_context(
        incident_frame=incident_frame,
        hypothesis_set=hypothesis_set,
        leading_assessment=leading_assessment,
        challenge_report=challenge_report,
    )

    system_prompt = """
You are the Root Cause & Remediation Agent for an autonomous network
investigation team.

A leading hypothesis has already been selected deterministically —
you do NOT choose among hypotheses. Your job is:

1. Explain, in plain language a network engineer would write in an
   incident report, why the leading hypothesis is the root cause,
   grounded only in the supplied evidence.
2. If a challenger_verdict is present and its challenge_outcome is
   "disputed", you must explicitly address the dispute in your
   explanation — either explain why the original evidence still
   holds despite the dispute, or note it as an open risk in
   residual_risk. Do NOT ignore a disputed challenge.
3. Propose concrete, ordered remediation steps. Each step needs a
   risk_level ("low", "medium", "high") reflecting the risk of
   performing that action, not the risk of the incident itself.
4. State any residual_risk: what could still be wrong even if this
   remediation is applied, given any gaps the evidence has.

Critical rules:

- Do NOT propose remediation steps that are not grounded in the
  evidence (e.g. do not suggest replacing hardware if the evidence
  only shows a logical state, not a confirmed physical fault).
- Do NOT invent additional hypotheses or entities.
- Prefer the least invasive remediation that addresses the
  evidenced cause; do not escalate to drastic steps without
  justification.
- If challenger_verdict shows challenge_outcome "disputed", you
  MUST populate challenge_acknowledged with how you addressed it.
  If there is no challenger_verdict or the outcome is "upheld",
  set challenge_acknowledged to a brief note saying no unresolved
  dispute exists.

Return ONLY a Python dictionary with exactly this shape:

{
    "root_cause_explanation": str,
    "remediation_steps": [
        {
            "action": str,
            "rationale": str,
            "risk_level": str
        }
    ],
    "residual_risk": str,
    "challenge_acknowledged": str
}
"""

    user_prompt = f"""
Investigation context:

{context}

Explain the root cause and recommend remediation.
"""

    response = llm_call(
        agent_name=agent_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.2,
    )

    print("Root cause remediation LLM call completed")

    content = response["content"]

    if not content:
        raise ValueError(
            "Root Cause & Remediation Agent returned empty LLM content."
        )

    parsed = parse_or_repair_agent_response(
        raw_output=content,
        expected_schema=ROOT_CAUSE_RECOMMENDATION_SCHEMA,
        agent_name=agent_name,
    )

    remediation_steps = []

    for index, item in enumerate(parsed.get("remediation_steps", []), start=1):
        step = RemediationStep(
            step_number=index,
            action=item["action"],
            rationale=item.get("rationale", ""),
            risk_level=item.get("risk_level", "medium"),
        )
        step.validate()
        remediation_steps.append(step)

    recommendation = RootCauseRecommendation(
        incident_id=incident_frame.incident_id,
        status="root_cause_identified",
        hypothesis_id=leading_assessment.hypothesis_id,
        root_cause_explanation=parsed.get("root_cause_explanation", ""),
        confidence=directional_confidence(leading_assessment),
        remediation_steps=remediation_steps,
        residual_risk=parsed.get("residual_risk"),
        challenge_acknowledged=parsed.get("challenge_acknowledged"),
    )

    recommendation.validate()

    return recommendation

#11. INVESTIGATION ORCHESTRATOR

In [85]:
# ============================================================
# INVESTIGATION RUN RESULT
# ============================================================

@dataclass
class InvestigationRunResult:
    """
    Complete output of one end-to-end investigation run.

    This bundles every intermediate artifact so failures can be
    traced back to the stage that produced them, rather than only
    seeing the final assessments.
    """

    investigation_id: str
    status: InvestigationStatus

    evidence_graph_summary: Dict[str, int]
    incident_frame: Optional[IncidentFrame] = None
    hypothesis_set: Optional[HypothesisSet] = None
    evidence_plan: Optional[EvidencePlan] = None
    evidence_results: List[EvidenceTestResult] = field(default_factory=list)
    evidence_analysis: Optional[EvidenceAnalysis] = None
    challenge_report: Optional[ChallengeReport] = None
    root_cause_recommendation: Optional[RootCauseRecommendation] = None

    failed_stage: Optional[str] = None
    error: Optional[str] = None

    def to_dict(self) -> Dict[str, Any]:
        return {
            "investigation_id": self.investigation_id,
            "status": self.status.value,
            "evidence_graph_summary": self.evidence_graph_summary,
            "incident_frame": (
                self.incident_frame.to_dict()
                if self.incident_frame else None
            ),
            "hypothesis_set": (
                self.hypothesis_set.to_dict()
                if self.hypothesis_set else None
            ),
            "evidence_plan": (
                self.evidence_plan.to_dict()
                if self.evidence_plan else None
            ),
            "evidence_results": [
                r.to_dict() for r in self.evidence_results
            ],
            "evidence_analysis": (
                self.evidence_analysis.to_dict()
                if self.evidence_analysis else None
            ),
            "challenge_report": (
                self.challenge_report.to_dict()
                if self.challenge_report else None
            ),
            "root_cause_recommendation": (
                self.root_cause_recommendation.to_dict()
                if self.root_cause_recommendation else None
            ),
            "failed_stage": self.failed_stage,
            "error": self.error,
        }

##INVESTIGATION ORCHESTRATOR

In [86]:
# ============================================================
# INVESTIGATION ORCHESTRATOR
# ============================================================

@observe(name="run_investigation")
def run_investigation(
    investigation_config: NetworkInvestigationConfig,
) -> InvestigationRunResult:
    """
    Run one full investigation pass:

      load evidence
        -> frame incident
        -> generate hypotheses
        -> plan evidence tests
        -> execute tests
        -> analyze evidence

    This is a single pass, not the iterative loop implied by
    InvestigationPolicy.maximum_reasoning_iterations. Looping,
    Hypothesis Challenger, and Root Cause & Remediation are not
    wired in yet — this function is the seam where they'll attach.

    Any stage failure is caught and reported with the stage name,
    rather than letting one exception obscure where the pipeline
    broke.
    """

    investigation_id = investigation_config.investigation_id
    evidence_graph = EvidenceGraph()

    # ------------------------------------------------------
    # Stage 0: Load evidence
    # ------------------------------------------------------

    try:
        load_result = load_investigation_evidence_executor(
            investigation_config=investigation_config,
            evidence_graph=evidence_graph,
        )
        evidence_graph = load_result["evidence_graph"]

    except Exception as exc:
        return InvestigationRunResult(
            investigation_id=investigation_id,
            status=InvestigationStatus.CREATED,
            evidence_graph_summary=evidence_graph.summary(),
            failed_stage="load_evidence",
            error=str(exc),
        )

    print(f"✓ Evidence loaded: {evidence_graph.summary()}")

    topology_capability = TopologyCapability()
    reachability_capability = ReachabilityCapability()
    network_state_capability = NetworkStateCapability()
    vendor_alert_capability = VendorAlertCapability()
    deep_diagnostics_capability = DeepDiagnosticsCapability()

    # ------------------------------------------------------
    # Stage 1: Frame the incident
    # ------------------------------------------------------

    try:
        incident_frame = incident_framing_agent(
            investigation_config=investigation_config,
            evidence_graph=evidence_graph,
            topology_capability=topology_capability,
        )
        incident_frame.validate()

    except Exception as exc:
        return InvestigationRunResult(
            investigation_id=investigation_id,
            status=InvestigationStatus.IN_PROGRESS,
            evidence_graph_summary=evidence_graph.summary(),
            failed_stage="incident_framing",
            error=str(exc),
        )

    print(f"✓ Incident framed: {incident_frame.investigation_question}")

    # ------------------------------------------------------
    # Stage 2: Generate hypotheses
    # ------------------------------------------------------

    try:
        hypothesis_set = hypothesis_generator_agent(
            incident_frame=incident_frame,
            evidence_graph=evidence_graph,
            topology_capability=topology_capability,
        )
        hypothesis_set.validate()

    except Exception as exc:
        return InvestigationRunResult(
            investigation_id=investigation_id,
            status=InvestigationStatus.IN_PROGRESS,
            evidence_graph_summary=evidence_graph.summary(),
            incident_frame=incident_frame,
            failed_stage="hypothesis_generation",
            error=str(exc),
        )

    print(f"✓ {len(hypothesis_set.hypotheses)} hypotheses generated")


    # ------------------------------------------------------
    # Stages 3-6: Plan, execute, analyze, and challenge evidence,
    # looping until a hypothesis both clears the confidence bar
    # AND survives the Challenger, or until max_iterations runs out.
    # ------------------------------------------------------

    policy = investigation_config.investigation_policy
    max_iterations = policy.maximum_reasoning_iterations

    evidence_plan = None
    evidence_results: List[EvidenceTestResult] = []
    evidence_analysis = None
    challenge_report = None
    round_number = 0

    for round_number in range(1, max_iterations + 1):

        # ------------------------------------------------------
        # Stage 3: Plan evidence tests
        # ------------------------------------------------------

        try:
            evidence_plan = evidence_planning_agent(
                incident_frame=incident_frame,
                hypothesis_set=hypothesis_set,
                round_number=round_number,
                prior_evidence_analysis=evidence_analysis,  # None on round 1
                prior_challenge_report=challenge_report,    # None on round 1
            )
            evidence_plan.validate()

        except Exception as exc:
            return InvestigationRunResult(
                investigation_id=investigation_id,
                status=InvestigationStatus.IN_PROGRESS,
                evidence_graph_summary=evidence_graph.summary(),
                incident_frame=incident_frame,
                hypothesis_set=hypothesis_set,
                evidence_results=evidence_results,
                failed_stage=f"evidence_planning_round_{round_number}",
                error=str(exc),
            )

        print(f"✓ Round {round_number}: {len(evidence_plan.tests)} evidence tests planned")

        # ------------------------------------------------------
        # Stage 4: Execute evidence tests (deterministic)
        # ------------------------------------------------------

        try:
            round_results = execute_evidence_plan(
                evidence_plan=evidence_plan,
                evidence_graph=evidence_graph,
                topology_capability=topology_capability,
                reachability_capability=reachability_capability,
                network_state_capability=network_state_capability,
                vendor_alert_capability=vendor_alert_capability,
                deep_diagnostics_capability=deep_diagnostics_capability,
            )

        except Exception as exc:
            return InvestigationRunResult(
                investigation_id=investigation_id,
                status=InvestigationStatus.WAITING_FOR_EVIDENCE,
                evidence_graph_summary=evidence_graph.summary(),
                incident_frame=incident_frame,
                hypothesis_set=hypothesis_set,
                evidence_plan=evidence_plan,
                evidence_results=evidence_results,
                failed_stage=f"evidence_execution_round_{round_number}",
                error=str(exc),
            )

        evidence_results = evidence_results + round_results

        print(f"✓ Round {round_number}: {len(round_results)} evidence tests executed")

        # ------------------------------------------------------
        # Stage 5: Analyze evidence against hypotheses
        # ------------------------------------------------------

        try:
            evidence_analysis = evidence_analyst_agent(
                incident_frame=incident_frame,
                hypothesis_set=hypothesis_set,
                evidence_plan=evidence_plan,
                evidence_results=evidence_results,  # cumulative across rounds
            )
            evidence_analysis.validate()

        except Exception as exc:
            return InvestigationRunResult(
                investigation_id=investigation_id,
                status=InvestigationStatus.WAITING_FOR_EVIDENCE,
                evidence_graph_summary=evidence_graph.summary(),
                incident_frame=incident_frame,
                hypothesis_set=hypothesis_set,
                evidence_plan=evidence_plan,
                evidence_results=evidence_results,
                failed_stage=f"evidence_analysis_round_{round_number}",
                error=str(exc),
            )

        print(f"✓ Round {round_number}: evidence analysis complete")
        for a in evidence_analysis.assessments:
            print(f"    {a.hypothesis_id}: {a.status.value} (confidence={a.confidence})")

        leakage_violations = check_evidence_analysis_leakage(evidence_analysis)
        if leakage_violations:
            print(f"⚠ Round {round_number}: Evidence Analyst contract violations detected:")
            for v in leakage_violations:
                print(f"    {v}")

        leading_assessment = select_leading_hypothesis(
            evidence_analysis=evidence_analysis,
            minimum_confidence=policy.minimum_diagnosis_confidence,
        )

        # If review is disabled, use the old, simpler exit condition and
        # skip the Challenger entirely.
        if not policy.require_challenger_review:
            if leading_assessment is not None:
                break

            if round_number < max_iterations:
                print(f"⚠ Round {round_number}: no hypothesis cleared the confidence bar — planning another round")

            continue

        # ------------------------------------------------------
        # Stage 6: Challenge the leading hypothesis
        # ------------------------------------------------------

        try:
            challenge_report = hypothesis_challenger_agent(
                incident_frame=incident_frame,
                hypothesis_set=hypothesis_set,
                evidence_analysis=evidence_analysis,
            )
            challenge_report.validate()

        except Exception as exc:
            return InvestigationRunResult(
                investigation_id=investigation_id,
                status=InvestigationStatus.IN_PROGRESS,
                evidence_graph_summary=evidence_graph.summary(),
                incident_frame=incident_frame,
                hypothesis_set=hypothesis_set,
                evidence_plan=evidence_plan,
                evidence_results=evidence_results,
                evidence_analysis=evidence_analysis,
                failed_stage=f"hypothesis_challenge_round_{round_number}",
                error=str(exc),
            )

        print(f"✓ Round {round_number}: hypothesis challenge complete")
        for c in challenge_report.challenges:
            print(f"    {c.hypothesis_id}: {c.challenge_outcome}")
        if challenge_report.systemic_concerns:
            print(f"    ⚠ {len(challenge_report.systemic_concerns)} systemic concern(s) raised")

        # Exit only if something cleared the confidence bar AND the
        # Challenger upheld it. Disputed or unresolved keeps looping,
        # using this round's challenger gaps to steer round N+1's plan.
        if leading_assessment is not None:
            leading_challenge = next(
                (
                    c for c in challenge_report.challenges
                    if c.hypothesis_id == leading_assessment.hypothesis_id
                ),
                None,
            )

            if leading_challenge is not None and leading_challenge.challenge_outcome == "upheld":
                break

        if round_number < max_iterations:
            print(f"⚠ Round {round_number}: leading hypothesis not confirmed — planning another round")


    # ------------------------------------------------------
    # Stage 7: Determine root cause and recommend remediation
    # ------------------------------------------------------

    if (
        investigation_config.investigation_policy.require_challenger_review
        and challenge_report is None
    ):
        return InvestigationRunResult(
            investigation_id=investigation_id,
            status=InvestigationStatus.WAITING_FOR_EVIDENCE,
            evidence_graph_summary=evidence_graph.summary(),
            incident_frame=incident_frame,
            hypothesis_set=hypothesis_set,
            evidence_plan=evidence_plan,
            evidence_results=evidence_results,
            evidence_analysis=evidence_analysis,
            challenge_report=challenge_report,
            failed_stage="root_cause_remediation",
            error=(
                "InvestigationPolicy.require_challenger_review is True "
                "but no challenge_report is available."
            ),
        )

    try:
        root_cause_recommendation = root_cause_remediation_agent(
            incident_frame=incident_frame,
            hypothesis_set=hypothesis_set,
            evidence_analysis=evidence_analysis,
            challenge_report=challenge_report,
            investigation_policy=investigation_config.investigation_policy,
        )

    except Exception as exc:
        return InvestigationRunResult(
            investigation_id=investigation_id,
            status=InvestigationStatus.IN_PROGRESS,
            evidence_graph_summary=evidence_graph.summary(),
            incident_frame=incident_frame,
            hypothesis_set=hypothesis_set,
            evidence_plan=evidence_plan,
            evidence_results=evidence_results,
            evidence_analysis=evidence_analysis,
            challenge_report=challenge_report,
            failed_stage="root_cause_remediation",
            error=str(exc),
        )

    if root_cause_recommendation.status == "insufficient_evidence":
        print("⚠ No hypothesis met minimum_diagnosis_confidence — investigation inconclusive")
        final_status = InvestigationStatus.INCONCLUSIVE
    else:
        print(
            f"✓ Root cause identified: {root_cause_recommendation.hypothesis_id} "
            f"(confidence={root_cause_recommendation.confidence})"
        )
        print(
            f"    {len(root_cause_recommendation.remediation_steps)} "
            "remediation step(s) recommended"
        )
        final_status = InvestigationStatus.REMEDIATION_RECOMMENDED

    return InvestigationRunResult(
        investigation_id=investigation_id,
        status=final_status,
        evidence_graph_summary=evidence_graph.summary(),
        incident_frame=incident_frame,
        hypothesis_set=hypothesis_set,
        evidence_plan=evidence_plan,
        evidence_results=evidence_results,
        evidence_analysis=evidence_analysis,
        challenge_report=challenge_report,
        root_cause_recommendation=root_cause_recommendation,
    )

    # ------------------------------------------------------
    # Done (pipeline currently stops here
    # Root Cause agent & remediation agent not wired in yet)
    # ------------------------------------------------------

    return InvestigationRunResult(
        investigation_id=investigation_id,
        status=InvestigationStatus.IN_PROGRESS,
        evidence_graph_summary=evidence_graph.summary(),
        incident_frame=incident_frame,
        hypothesis_set=hypothesis_set,
        evidence_plan=evidence_plan,
        evidence_results=evidence_results,
        evidence_analysis=evidence_analysis,
        challenge_report=challenge_report,
    )

testing the orchestrator

In [109]:
run_result = run_investigation(
    investigation_config=nika_simple_bgp_config,
)

pprint.pprint(run_result.to_dict())

✓ Loaded nika_simple_bgp_topology using NIKASimpleBGPAdapter
✓ Loaded nika_simple_bgp_operational using NIKASimpleBGPOperationalAdapter
✓ Loaded pagerduty_simple_bgp_incident using PagerDutySimpleBGPIncidentAdapter
✓ Evidence loaded: {'entities': 4, 'relationships': 3, 'observations': 5, 'events': 1, 'changes': 0}
llm call completed
✓ Incident framed: Why is connectivity between pc1 and pc2 failing, and what is the state of the network path through router1 and router2?
llm call completed
Hypothesis generation LLM call completed
✓ 4 hypotheses generated
llm call completed
Evidence planning LLM call completed
✓ Round 1: 6 evidence tests planned
✓ Round 1: 6 evidence tests executed
llm call completed
Evidence analysis LLM call completed
⚠ evidence_analyst_agent returned malformed structured output. Attempting format repair.
llm call completed
✓ Round 1: evidence analysis complete
    nika-simple-bgp-link-down-001-incident-frame-h1: supported (confidence=0.95)
    nika-simple-bgp-link-down

Summarizing the run result output

In [110]:
def print_run_summary(run_result: InvestigationRunResult) -> None:
    print(f"Status: {run_result.status.value}")
    print(f"Failed stage: {run_result.failed_stage}")
    print(f"Error: {run_result.error}")
    print()

    if run_result.evidence_analysis:
        print("Final evidence_analysis:")
        for a in run_result.evidence_analysis.assessments:
            print(f"    {a.hypothesis_id}: {a.status.value} (confidence={a.confidence})")
        print()

    if run_result.challenge_report:
        print("Final challenge_report:")
        for c in run_result.challenge_report.challenges:
            print(f"    {c.hypothesis_id}: {c.challenge_outcome}")
        print()

    print("Rounds seen in evidence_results (by test_id prefix):")
    rounds_seen = sorted({
        r.test_id.split("-test-")[0].split("-r")[-1]
        for r in run_result.evidence_results
    })
    print(f"    {rounds_seen}")
    print()

    capabilities_used = sorted({r.capability for r in run_result.evidence_results})
    print(f"Capabilities used across all rounds: {capabilities_used}")
    print()

    if run_result.root_cause_recommendation:
        rec = run_result.root_cause_recommendation
        print(f"Root cause recommendation: {rec.hypothesis_id} (confidence={rec.confidence})")
        print(f"Status: {rec.status}")


print_run_summary(run_result)

Status: remediation_recommended
Failed stage: None
Error: None

Final evidence_analysis:
    nika-simple-bgp-link-down-001-incident-frame-h1: supported (confidence=0.95)
    nika-simple-bgp-link-down-001-incident-frame-h2: rejected (confidence=0.05)
    nika-simple-bgp-link-down-001-incident-frame-h3: rejected (confidence=0.02)
    nika-simple-bgp-link-down-001-incident-frame-h4: rejected (confidence=0.01)

Final challenge_report:
    nika-simple-bgp-link-down-001-incident-frame-h1: upheld
    nika-simple-bgp-link-down-001-incident-frame-h2: upheld

Rounds seen in evidence_results (by test_id prefix):
    ['1']

Capabilities used across all rounds: ['deep_diagnostics', 'network_state', 'reachability']

Root cause recommendation: nika-simple-bgp-link-down-001-incident-frame-h1 (confidence=0.95)
Status: root_cause_identified


##Persisting Investigation Results & Efficency Metrics

In [89]:
# ============================================================
# PERSIST INVESTIGATION RESULT
# ============================================================

def save_investigation_result(
    run_result: InvestigationRunResult,
    investigation_config: NetworkInvestigationConfig,
) -> Path:
    """
    Persist a completed investigation run to
    investigation_config.report_path as JSON.

    Creates the investigation's output directory if it doesn't
    exist yet. Overwrites any existing report for this
    investigation_id — callers who want history across multiple
    runs of the same investigation should use append_investigation_history
    (see below) instead of relying on this file alone.
    """

    output_dir = investigation_config.investigation_output_dir
    output_dir.mkdir(parents=True, exist_ok=True)

    report_path = investigation_config.report_path

    with open(report_path, "w") as f:
        json.dump(run_result.to_dict(), f, indent=2, default=str)

    print(f"✓ Investigation result saved to {report_path}")

    return report_path

testing the persistance of investigation result

In [90]:
# testing save_investigation_result

saved_path = save_investigation_result(
    run_result=run_result,
    investigation_config=nika_simple_bgp_config,
)

✓ Investigation result saved to /content/network_investigations/nika-simple-bgp-link-down-001/investigation_report.json


###Efficency Metrics

In [91]:
# ============================================================
# EFFICIENCY METRICS
# ============================================================

@dataclass
class EfficiencyMetrics:
    """
    Summary of LLM usage for one investigation run.

    Computed from a slice of the global LLM_USAGE list — the
    caller is responsible for capturing the right slice (see
    compute_efficiency_metrics usage example). This does not
    itself know which investigation a call belongs to; LLM_USAGE
    has no investigation_id field, so scoping is done by the
    caller via list-slicing around the run_investigation() call,
    not by anything in this function.
    """

    total_calls: int
    total_input_tokens: int
    total_output_tokens: int
    total_tokens: int

    calls_by_agent: Dict[str, int] = field(default_factory=dict)
    tokens_by_agent: Dict[str, int] = field(default_factory=dict)

    repair_call_count: int = 0

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


def compute_efficiency_metrics(
    llm_usage_slice: List[Dict[str, Any]],
) -> EfficiencyMetrics:
    """
    Summarize a slice of LLM_USAGE into per-investigation metrics.

    Usage:

        usage_start = len(LLM_USAGE)
        run_result = run_investigation(
            investigation_config=nika_simple_bgp_config,
        )
        usage_end = len(LLM_USAGE)

        efficiency = compute_efficiency_metrics(
            LLM_USAGE[usage_start:usage_end]
        )

    repair_call_count counts entries whose agent name ends in
    "_format_repair" — repair_agent_dict_response tags its own
    llm_call with that suffix rather than reusing the parent
    agent's name, so repair calls are directly identifiable rather
    than inferred from call-count anomalies.
    """

    total_calls = len(llm_usage_slice)
    total_input_tokens = sum(e["input_tokens"] for e in llm_usage_slice)
    total_output_tokens = sum(e["output_tokens"] for e in llm_usage_slice)
    total_tokens = sum(e["total_tokens"] for e in llm_usage_slice)

    calls_by_agent: Dict[str, int] = {}
    tokens_by_agent: Dict[str, int] = {}

    for entry in llm_usage_slice:
        agent = entry["agent"]
        calls_by_agent[agent] = calls_by_agent.get(agent, 0) + 1
        tokens_by_agent[agent] = (
            tokens_by_agent.get(agent, 0) + entry["total_tokens"]
        )

    repair_call_count = sum(
        1 for entry in llm_usage_slice
        if entry["agent"].endswith("_format_repair")
    )

    return EfficiencyMetrics(
        total_calls=total_calls,
        total_input_tokens=total_input_tokens,
        total_output_tokens=total_output_tokens,
        total_tokens=total_tokens,
        calls_by_agent=calls_by_agent,
        tokens_by_agent=tokens_by_agent,
        repair_call_count=repair_call_count,
    )

testing efficency metrics against a fresh run

In [92]:
# testing efficiency metrics against a fresh run

usage_start = len(LLM_USAGE)

run_result = run_investigation(
    investigation_config=nika_simple_bgp_config,
)

usage_end = len(LLM_USAGE)

efficiency = compute_efficiency_metrics(
    LLM_USAGE[usage_start:usage_end]
)

pprint.pprint(efficiency.to_dict())

✓ Loaded nika_simple_bgp_topology using NIKASimpleBGPAdapter
✓ Loaded nika_simple_bgp_operational using NIKASimpleBGPOperationalAdapter
✓ Loaded pagerduty_simple_bgp_incident using PagerDutySimpleBGPIncidentAdapter
✓ Evidence loaded: {'entities': 4, 'relationships': 3, 'observations': 5, 'events': 1, 'changes': 0}
llm call completed
✓ Incident framed: Why is connectivity between pc1 and pc2 failing, and what is the state of the network path through router1 and router2?
llm call completed
Hypothesis generation LLM call completed
✓ 4 hypotheses generated
llm call completed
Evidence planning LLM call completed
✓ Round 1: 5 evidence tests planned
✓ Round 1: 5 evidence tests executed
llm call completed
Evidence analysis LLM call completed
{'calls_by_agent': {'evidence_analyst_agent': 1,
                    'evidence_planning_agent': 1,
                    'hypothesis_generator_agent': 1,
                    'incident_framing_agent': 1},
 'repair_call_count': 0,
 'tokens_by_agent': {'evidenc

throwaway test

In [93]:
# ============================================================
# THROWAWAY TEST: force the loop to run multiple rounds
# by setting an unreachable confidence threshold.
# Not representative of a real investigation — diagnostic only.
# ============================================================

import dataclasses

forced_policy = dataclasses.replace(
    nika_simple_bgp_config.investigation_policy,
    minimum_diagnosis_confidence=0.999999,  # unreachable — forces every round to fail the bar
)

forced_config = dataclasses.replace(
    nika_simple_bgp_config,
    investigation_policy=forced_policy,
)

loop_test_result = run_investigation(
    investigation_config=forced_config,
)

print()
print("=" * 60)
print(f"Final status: {loop_test_result.status.value}")
print(f"Failed stage: {loop_test_result.failed_stage}")
print(f"Total evidence_results across all rounds: {len(loop_test_result.evidence_results)}")
print()

test_ids = [r.test_id for r in loop_test_result.evidence_results]
print("All test_ids across rounds:")
for tid in test_ids:
    print(f"    {tid}")

duplicate_test_ids = [t for t in set(test_ids) if test_ids.count(t) > 1]
print()
print(f"Duplicate test_ids found: {duplicate_test_ids if duplicate_test_ids else 'none'}")

✓ Loaded nika_simple_bgp_topology using NIKASimpleBGPAdapter
✓ Loaded nika_simple_bgp_operational using NIKASimpleBGPOperationalAdapter
✓ Loaded pagerduty_simple_bgp_incident using PagerDutySimpleBGPIncidentAdapter
✓ Evidence loaded: {'entities': 4, 'relationships': 3, 'observations': 5, 'events': 1, 'changes': 0}
llm call completed
✓ Incident framed: Why is connectivity between pc1 and pc2 failing, and what is the state of the network path through router1 and router2?
llm call completed
Hypothesis generation LLM call completed
✓ 4 hypotheses generated
llm call completed
Evidence planning LLM call completed
✓ Round 1: 7 evidence tests planned
✓ Round 1: 7 evidence tests executed
llm call completed
Evidence analysis LLM call completed
⚠ evidence_analyst_agent returned malformed structured output. Attempting format repair.
llm call completed
✓ Round 1: evidence analysis complete
    nika-simple-bgp-link-down-001-incident-frame-h1: supported (confidence=0.95)
    nika-simple-bgp-link-down

#12. VALIDATION

In [94]:
EVIDENCE_ANALYST_BANNED_PHRASES = [
    "root cause",
    "confirmed",
    "definitively",
]


def check_evidence_analysis_leakage(
    evidence_analysis: EvidenceAnalysis,
) -> List[str]:
    """
    Deterministic guardrail: flag any assessment text that leaks
    root-cause language the Evidence Analyst is contractually
    forbidden from using.

    Returns a list of violation descriptions. An empty list means
    no violations were found.
    """

    violations = []

    text_fields = (
        "rationale",
        "supporting_evidence",
        "contradicting_evidence",
        "missing_evidence",
    )

    for assessment in evidence_analysis.assessments:
        for field_name in text_fields:
            value = getattr(assessment, field_name)

            # normalize to a single string to search
            if isinstance(value, list):
                haystack = " ".join(value).lower()
            else:
                haystack = str(value).lower()

            for phrase in EVIDENCE_ANALYST_BANNED_PHRASES:
                if phrase in haystack:
                    violations.append(
                        f"{assessment.hypothesis_id}.{field_name} "
                        f"contains banned phrase '{phrase}'"
                    )

    return violations